In [ ]:
!pip install -U anthropic pandas tqdm python-dotenv

In [1]:
from __future__ import annotations

import os
import re
import json
import time
import uuid
import hashlib
import datetime as dt
from pathlib import Path
from typing import Any, Dict, List, Optional, Tuple

import pandas as pd
from tqdm.auto import tqdm
from dotenv import load_dotenv
import anthropic

load_dotenv()

# assert os.getenv("ANTHROPIC_API_KEY"), "Missing ANTHROPIC_API_KEY. Put it in your environment or .env file."
client = anthropic.Anthropic(api_key="")

# -----------------------------
# Experiment configuration
# -----------------------------
PROVIDER = "anthropic"
MODEL_NAME = "claude-sonnet-4-6"
TEMPERATURE = 1.0

# This notebook only collects the new task settings.
# Keep this ID stable so later analysis can identify this task set.
TASK_SET_ID = "taskset_b_additional_prompts"

# Anthropic does not use OpenAI-style reasoning_effort/text_verbosity.
# Do not enable extended thinking for this constrained creative-generation task.
ANTHROPIC_THINKING = None

# Prompt caching is available on active Claude models, but is off by default here.
# Batch processing is the main cost-control mechanism for this collection.
ANTHROPIC_ENABLE_PROMPT_CACHING = False
ANTHROPIC_CACHE_CONTROL = {"type": "ephemeral"}  # only used if enabled

N_BASE_AGENTS = 150
N_DYADS = 75
N_TRIADS = 50

MAX_OUTPUT_TOKENS_BY_FAMILY = {
    "slogan": 60,
    "aut": 120,
    "story": 700,
}

RUN_ID = dt.datetime.now().strftime("%Y%m%d_%H%M%S") + "__" + uuid.uuid4().hex[:8]

DATA_ROOT = (
    Path("ai_data")
    / "deflect_creativity"
    / PROVIDER
    / f"model_{MODEL_NAME}"
    / TASK_SET_ID
    / f"run_{RUN_ID}"
)

DIRS = {
    "metadata": DATA_ROOT / "00_metadata",
    "round1_plans": DATA_ROOT / "01_round1" / "plans",
    "round1_batch_inputs": DATA_ROOT / "01_round1" / "batch_inputs",
    "round1_manifests": DATA_ROOT / "01_round1" / "manifests",
    "round1_raw_outputs": DATA_ROOT / "01_round1" / "raw_outputs",
    "round1_parsed": DATA_ROOT / "01_round1" / "parsed",
    "round2_plans": DATA_ROOT / "02_round2" / "plans",
    "round2_batch_inputs": DATA_ROOT / "02_round2" / "batch_inputs",
    "round2_manifests": DATA_ROOT / "02_round2" / "manifests",
    "round2_raw_outputs": DATA_ROOT / "02_round2" / "raw_outputs",
    "round2_parsed": DATA_ROOT / "02_round2" / "parsed",
    "compiled": DATA_ROOT / "03_compiled",
}

for d in DIRS.values():
    d.mkdir(parents=True, exist_ok=True)

print("Run directory:")
print(DATA_ROOT)

Run directory:
ai_data/deflect_creativity/anthropic/model_claude-sonnet-4-6/taskset_b_additional_prompts/run_20260518_095240__3837589d


In [2]:
def now_iso() -> str:
    return dt.datetime.now(dt.timezone.utc).isoformat()


def write_json(path: Path, obj: Any) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    with open(path, "w", encoding="utf-8") as f:
        json.dump(obj, f, ensure_ascii=False, indent=2)


def read_json(path: Path) -> Any:
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)


def append_jsonl(path: Path, record: dict) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    with open(path, "a", encoding="utf-8") as f:
        f.write(json.dumps(record, ensure_ascii=False) + "\n")


def read_jsonl(path: Path) -> list[dict]:
    records = []
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if line:
                records.append(json.loads(line))
    return records


def safe_slug(text: str) -> str:
    text = str(text).lower()
    text = re.sub(r"[^a-z0-9]+", "_", text)
    text = re.sub(r"_+", "_", text).strip("_")
    return text


def stable_hash(text: str, n: int = 16) -> str:
    return hashlib.sha256(text.encode("utf-8")).hexdigest()[:n]


def clean_model_text(text: Optional[str]) -> Optional[str]:
    if text is None:
        return None
    text = str(text).strip()
    if len(text) >= 2 and ((text[0] == text[-1] == '"') or (text[0] == text[-1] == "'")):
        text = text[1:-1].strip()
    return text


def to_jsonable(obj: Any) -> Any:
    if hasattr(obj, "model_dump"):
        return obj.model_dump(mode="json")
    if hasattr(obj, "dict"):
        return obj.dict()
    return obj

In [3]:
run_config = {
    "provider": PROVIDER,
    "model_name": MODEL_NAME,
    "task_set_id": TASK_SET_ID,
    "temperature": TEMPERATURE,
    "anthropic_thinking": ANTHROPIC_THINKING,
    "anthropic_enable_prompt_caching": ANTHROPIC_ENABLE_PROMPT_CACHING,
    "anthropic_cache_control": ANTHROPIC_CACHE_CONTROL if ANTHROPIC_ENABLE_PROMPT_CACHING else None,
    "n_base_agents": N_BASE_AGENTS,
    "n_dyads": N_DYADS,
    "n_triads": N_TRIADS,
    "max_output_tokens_by_family": MAX_OUTPUT_TOKENS_BY_FAMILY,
    "run_id": RUN_ID,
    "data_root": str(DATA_ROOT),
    "created_at_utc": now_iso(),
}

config_path = DIRS["metadata"] / f"experiment_config__{TASK_SET_ID}__{RUN_ID}.json"
write_json(config_path, run_config)

config_path

PosixPath('ai_data/deflect_creativity/anthropic/model_claude-sonnet-4-6/taskset_b_additional_prompts/run_20260518_095240__3837589d/00_metadata/experiment_config__taskset_b_additional_prompts__20260518_095240__3837589d.json')

In [4]:
TASK_SETTINGS = [
    {
        "task_id": "slogan_blood_donation",
        "task_family": "slogan",
        "task_label": "Blood donation slogan",
        "task_prompt_key": "blood_donation",
    },
    {
        "task_id": "aut_key",
        "task_family": "aut",
        "task_label": "AUT key",
        "task_prompt_key": "key",
        "object": "key",
        "common_use": "used to open a lock",
    },
    {
        "task_id": "aut_wooden_pencil",
        "task_family": "aut",
        "task_label": "AUT wooden pencil",
        "task_prompt_key": "wooden_pencil",
        "object": "wooden pencil",
        "common_use": "used for writing",
    },
    {
        "task_id": "aut_automobile_tire",
        "task_family": "aut",
        "task_label": "AUT automobile tire",
        "task_prompt_key": "automobile_tire",
        "object": "automobile tire",
        "common_use": "used on the wheel of an automobile",
    },
    {
        "task_id": "story_horror",
        "task_family": "story",
        "task_label": "Horror story",
        "task_prompt_key": "horror",
    },
    {
        "task_id": "story_life_last_seconds",
        "task_family": "story",
        "task_label": "Life and last seconds story",
        "task_prompt_key": "life_last_seconds",
    },
]

STRATEGIES = ["vanilla", "diverge"]
CONDITIONS = ["base", "dyad", "triad"]

tasks_df = pd.DataFrame(TASK_SETTINGS)
tasks_df

,task_id,task_family,task_label,task_prompt_key,object,common_use
0,slogan_blood_donation,slogan,Blood donation slogan,blood_donation,NaN,NaN
1,aut_key,aut,AUT key,key,key,used to open a lock
2,aut_wooden_pencil,aut,AUT wooden pencil,wooden_pencil,wooden pencil,used for writing
3,aut_automobile_tire,aut,AUT automobile tire,automobile_tire,automobile tire,used on the wheel of an automobile
4,story_horror,story,Horror story,horror,NaN,NaN
5,story_life_last_seconds,story,Life and last seconds story,life_last_seconds,NaN,NaN


In [5]:
SYSTEM_INSTRUCTIONS = (
    "You are participating in a controlled creativity experiment. "
    "Follow the task instructions exactly. Return exactly one response. "
    "Do not explain your reasoning. Do not include commentary before or after the response."
)


def strategy_block(strategy: str) -> str:
    if strategy == "vanilla":
        return (
            "Creativity goal:\n"
            "- Make the response novel and appropriate for the task."
        )

    if strategy == "diverge":
        return (
            "Creativity goal:\n"
            "- Make the response novel and appropriate for the task.\n"
            "- Try to make it stand out from other responses that might be generated for this same task."
        )

    raise ValueError(f"Unknown strategy: {strategy}")


def base_task_prompt(task: dict) -> str:
    task_id = task["task_id"]

    if task_id == "slogan_blood_donation":
        return (
            "You are part of the communications team at a nonprofit organization preparing a campaign "
            "to encourage blood donation.\n\n"
            "Generate exactly one campaign slogan for this blood donation campaign.\n\n"
            "Requirements:\n"
            "- The slogan must not exceed 6 words.\n"
            "- The slogan must be written in English.\n"
            "- You may assume any detail about the campaign.\n"
            "- Do not list multiple slogans.\n"
            "- Return only the slogan text."
        )

    if task_id in {"aut_key", "aut_wooden_pencil", "aut_automobile_tire"}:
        return (
            "You are participating in a creativity task.\n\n"
            f"Object: {task['object']}\n"
            f"Common use to avoid: {task['common_use']}\n\n"
            "Generate exactly one unusual, novel, and plausible alternative use for the object or one of its parts.\n\n"
            "Requirements:\n"
            "- Do not use the common use.\n"
            "- Do not list multiple uses.\n"
            "- The response must be written in English.\n"
            "- Return only the alternative use as a short phrase or one sentence."
        )

    if task_id == "story_horror":
        return (
            "You are participating in a creative writing task.\n\n"
            "Write exactly one short horror story designed to chill the bones.\n\n"
            "Requirements:\n"
            "- The story must be exactly 8 sentences long.\n"
            "- The story must be written in English.\n"
            "- The story must be appropriate for a teenage and young adult audience, approximately ages 15 to 24.\n"
            "- Do not provide multiple story ideas.\n"
            "- Do not summarize the story.\n"
            "- Return only the story."
        )

    if task_id == "story_life_last_seconds":
        return (
            "You are participating in a creative writing task.\n\n"
            "Write exactly one story in 8 sentences. The first sentence must describe 100 years of a character's life. "
            "The next 7 sentences must describe the last 10 seconds of that character's life.\n\n"
            "Requirements:\n"
            "- The story must be exactly 8 sentences long.\n"
            "- The story must be written in English.\n"
            "- The story must be appropriate for a teenage and young adult audience, approximately ages 15 to 24.\n"
            "- Do not number or label the sentences.\n"
            "- Do not state which sentence does what.\n"
            "- Return only the story as one paragraph."
        )

    raise ValueError(f"Unknown task_id: {task_id}")


def build_round1_prompt(task: dict, strategy: str) -> str:
    return base_task_prompt(task) + "\n\n" + strategy_block(strategy)


def build_round2_context(
    strategy: str,
    condition: str,
    self_round1: str,
    peer_round1_texts: list[str],
) -> str:
    if condition == "base":
        context = (
            f'Previous response from your first round:\n'
            f'"{self_round1}"\n\n'
        )

    elif condition == "dyad":
        assert len(peer_round1_texts) == 1
        context = (
            f'Previous response from your first round:\n'
            f'"{self_round1}"\n\n'
            f'Previous response from another agent in the same first round:\n'
            f'"{peer_round1_texts[0]}"\n\n'
        )

    elif condition == "triad":
        assert len(peer_round1_texts) == 2
        context = (
            f'Previous response from your first round:\n'
            f'"{self_round1}"\n\n'
            f'Previous responses from two other agents in the same first round:\n'
            f'1. "{peer_round1_texts[0]}"\n'
            f'2. "{peer_round1_texts[1]}"\n\n'
        )

    else:
        raise ValueError(f"Unknown condition: {condition}")

    if strategy == "vanilla":
        return context + "Now generate one new response for the same task."

    if strategy == "diverge":
        return (
            context
            + "Now generate one new response for the same task. "
              "It should stand out from the previous response(s) shown above while still satisfying all task requirements."
        )

    raise ValueError(f"Unknown strategy: {strategy}")


def build_round2_prompt(
    task: dict,
    strategy: str,
    condition: str,
    self_round1: str,
    peer_round1_texts: list[str],
) -> str:
    return (
        base_task_prompt(task)
        + "\n\n"
        + strategy_block(strategy)
        + "\n\n"
        + build_round2_context(
            strategy=strategy,
            condition=condition,
            self_round1=self_round1,
            peer_round1_texts=peer_round1_texts,
        )
    )

In [6]:
def build_agent_roster() -> pd.DataFrame:
    rows = []

    for i in range(1, N_BASE_AGENTS + 1):
        rows.append({
            "condition": "base",
            "group_id": f"base_{i:03d}",
            "group_size": 1,
            "agent_index": 1,
            "agent_id": f"base_{i:03d}__a1",
        })

    for g in range(1, N_DYADS + 1):
        for a in [1, 2]:
            rows.append({
                "condition": "dyad",
                "group_id": f"dyad_{g:03d}",
                "group_size": 2,
                "agent_index": a,
                "agent_id": f"dyad_{g:03d}__a{a}",
            })

    for g in range(1, N_TRIADS + 1):
        for a in [1, 2, 3]:
            rows.append({
                "condition": "triad",
                "group_id": f"triad_{g:03d}",
                "group_size": 3,
                "agent_index": a,
                "agent_id": f"triad_{g:03d}__a{a}",
            })

    return pd.DataFrame(rows)


agents_df = build_agent_roster()

display(agents_df.groupby("condition").agg(
    n_agents=("agent_id", "count"),
    n_groups=("group_id", "nunique"),
    group_size=("group_size", "first"),
))

agents_df.head()

,n_agents,n_groups,group_size
condition,,,
base,150,150,1
dyad,150,75,2
triad,150,50,3


,condition,group_id,group_size,agent_index,agent_id
0,base,base_001,1,1,base_001__a1
1,base,base_002,1,1,base_002__a1
2,base,base_003,1,1,base_003__a1
3,base,base_004,1,1,base_004__a1
4,base,base_005,1,1,base_005__a1


In [7]:
def build_round1_plan() -> pd.DataFrame:
    rows = []

    for task in TASK_SETTINGS:
        for strategy in STRATEGIES:
            for _, agent in agents_df.iterrows():
                user_prompt = build_round1_prompt(task, strategy)

                request_basis = {
                    "provider": PROVIDER,
                    "model": MODEL_NAME,
                    "task_set_id": TASK_SET_ID,
                    "round": 1,
                    "task_id": task["task_id"],
                    "task_family": task["task_family"],
                    "strategy": strategy,
                    "condition": agent["condition"],
                    "group_id": agent["group_id"],
                    "agent_id": agent["agent_id"],
                    "agent_index": int(agent["agent_index"]),
                }

                request_key = "r1__" + stable_hash(json.dumps(request_basis, sort_keys=True), 24)

                rows.append({
                    **request_basis,
                    "request_key": request_key,
                    "system_instructions": SYSTEM_INSTRUCTIONS,
                    "user_prompt": user_prompt,
                    "temperature": TEMPERATURE,
                    "max_output_tokens": MAX_OUTPUT_TOKENS_BY_FAMILY[task["task_family"]],
                    "created_at_utc": now_iso(),
                })

    plan_df = pd.DataFrame(rows)

    if plan_df["request_key"].duplicated().any():
        dupes = plan_df[plan_df["request_key"].duplicated(keep=False)].sort_values("request_key")
        raise ValueError(f"Duplicate request_key detected:\n{dupes.head()}")

    return plan_df


round1_plan_df = build_round1_plan()

expected_round1 = len(TASK_SETTINGS) * len(STRATEGIES) * len(agents_df)

print("Expected Round 1 requests:", expected_round1)
print("Actual Round 1 requests:  ", len(round1_plan_df))

display(round1_plan_df.groupby(["task_id", "strategy", "condition"]).size().reset_index(name="n"))
round1_plan_df.head()

Expected Round 1 requests: 5400
Actual Round 1 requests:   5400


,task_id,strategy,condition,n
0,aut_automobile_tire,diverge,base,150
1,aut_automobile_tire,diverge,dyad,150
2,aut_automobile_tire,diverge,triad,150
3,aut_automobile_tire,vanilla,base,150
4,aut_automobile_tire,vanilla,dyad,150
5,aut_automobile_tire,vanilla,triad,150
6,aut_key,diverge,base,150
7,aut_key,diverge,dyad,150
8,aut_key,diverge,triad,150
9,aut_key,vanilla,base,150


,provider,model,task_set_id,round,task_id,task_family,strategy,condition,group_id,agent_id,agent_index,request_key,system_instructions,user_prompt,temperature,max_output_tokens,created_at_utc
0,anthropic,claude-sonnet-4-6,taskset_b_additional_prompts,1,slogan_blood_donation,slogan,vanilla,base,base_001,base_001__a1,1,r1__28010387f9345a90d2bedbcf,You are participating in a controlled creativi...,You are part of the communications team at a n...,1.0,60,2026-05-18T13:52:42.883270+00:00
1,anthropic,claude-sonnet-4-6,taskset_b_additional_prompts,1,slogan_blood_donation,slogan,vanilla,base,base_002,base_002__a1,1,r1__c1aafe0d4ad9bb1882ec8a65,You are participating in a controlled creativi...,You are part of the communications team at a n...,1.0,60,2026-05-18T13:52:42.883470+00:00
2,anthropic,claude-sonnet-4-6,taskset_b_additional_prompts,1,slogan_blood_donation,slogan,vanilla,base,base_003,base_003__a1,1,r1__2969b6ff93a8255bd4a90abe,You are participating in a controlled creativi...,You are part of the communications team at a n...,1.0,60,2026-05-18T13:52:42.883653+00:00
3,anthropic,claude-sonnet-4-6,taskset_b_additional_prompts,1,slogan_blood_donation,slogan,vanilla,base,base_004,base_004__a1,1,r1__b9f9c5dfcdedb51ee6c4df65,You are participating in a controlled creativi...,You are part of the communications team at a n...,1.0,60,2026-05-18T13:52:42.884028+00:00
4,anthropic,claude-sonnet-4-6,taskset_b_additional_prompts,1,slogan_blood_donation,slogan,vanilla,base,base_005,base_005__a1,1,r1__44aa09a83bc1622ef8d2ce00,You are participating in a controlled creativi...,You are part of the communications team at a n...,1.0,60,2026-05-18T13:52:42.884612+00:00


In [8]:
def make_anthropic_message_params(row: pd.Series) -> dict:
    params = {
        "model": MODEL_NAME,
        "max_tokens": int(row["max_output_tokens"]),
        "temperature": float(row["temperature"]),
        "system": row["system_instructions"],
        "messages": [
            {
                "role": "user",
                "content": row["user_prompt"],
            }
        ],
    }

    if ANTHROPIC_THINKING is not None:
        params["thinking"] = ANTHROPIC_THINKING

    if ANTHROPIC_ENABLE_PROMPT_CACHING:
        params["cache_control"] = ANTHROPIC_CACHE_CONTROL

    return params


def make_anthropic_batch_request_files(
    plan_df: pd.DataFrame,
    round_name: str,
    batch_input_dir: Path,
) -> tuple[list[dict], Path, Path]:
    timestamp = dt.datetime.now().strftime("%Y%m%d_%H%M%S")
    stem = f"{TASK_SET_ID}__{round_name}__{PROVIDER}__{MODEL_NAME}__{timestamp}"

    plan_path = batch_input_dir.parent / "plans" / f"{stem}__plan.csv"
    jsonl_path = batch_input_dir / f"{stem}__local_batch_requests.jsonl"

    plan_path.parent.mkdir(parents=True, exist_ok=True)
    jsonl_path.parent.mkdir(parents=True, exist_ok=True)

    if plan_path.exists() or jsonl_path.exists():
        raise FileExistsError("Refusing to overwrite an existing plan or batch-input file.")

    plan_df.to_csv(plan_path, index=False)

    requests = []
    with open(jsonl_path, "w", encoding="utf-8") as f:
        for _, row in plan_df.iterrows():
            request = {
                "custom_id": row["request_key"],
                "params": make_anthropic_message_params(row),
            }
            requests.append(request)
            f.write(json.dumps(request, ensure_ascii=False) + "\n")

    print(f"Wrote plan:          {plan_path}")
    print(f"Wrote local JSONL:   {jsonl_path}")
    print(f"Batch request count: {len(requests):,}")

    return requests, jsonl_path, plan_path


round1_requests, round1_jsonl_path, round1_plan_path = make_anthropic_batch_request_files(
    plan_df=round1_plan_df,
    round_name="round1",
    batch_input_dir=DIRS["round1_batch_inputs"],
)

round1_jsonl_path, round1_plan_path

Wrote plan:          ai_data/deflect_creativity/anthropic/model_claude-sonnet-4-6/taskset_b_additional_prompts/run_20260518_095240__3837589d/01_round1/plans/taskset_b_additional_prompts__round1__anthropic__claude-sonnet-4-6__20260518_095244__plan.csv
Wrote local JSONL:   ai_data/deflect_creativity/anthropic/model_claude-sonnet-4-6/taskset_b_additional_prompts/run_20260518_095240__3837589d/01_round1/batch_inputs/taskset_b_additional_prompts__round1__anthropic__claude-sonnet-4-6__20260518_095244__local_batch_requests.jsonl
Batch request count: 5,400


(PosixPath('ai_data/deflect_creativity/anthropic/model_claude-sonnet-4-6/taskset_b_additional_prompts/run_20260518_095240__3837589d/01_round1/batch_inputs/taskset_b_additional_prompts__round1__anthropic__claude-sonnet-4-6__20260518_095244__local_batch_requests.jsonl'),
 PosixPath('ai_data/deflect_creativity/anthropic/model_claude-sonnet-4-6/taskset_b_additional_prompts/run_20260518_095240__3837589d/01_round1/plans/taskset_b_additional_prompts__round1__anthropic__claude-sonnet-4-6__20260518_095244__plan.csv'))

In [9]:
def submit_anthropic_batch(
    requests: list[dict],
    round_name: str,
    plan_path: Path,
    local_jsonl_path: Path,
    manifest_dir: Path,
) -> dict:
    message_batch = client.messages.batches.create(requests=requests)
    batch_dump = to_jsonable(message_batch)

    batch_info = {
        "run_id": RUN_ID,
        "task_set_id": TASK_SET_ID,
        "round": round_name,
        "provider": PROVIDER,
        "model": MODEL_NAME,
        "message_batch": batch_dump,
        "batch_id": batch_dump.get("id"),
        "processing_status_at_submission": batch_dump.get("processing_status"),
        "submitted_at_utc": now_iso(),
        "local_jsonl_path": str(local_jsonl_path),
        "plan_path": str(plan_path),
        "data_root": str(DATA_ROOT),
    }

    manifest_path = (
        manifest_dir
        / f"{TASK_SET_ID}__{round_name}__{PROVIDER}__{MODEL_NAME}__batch_manifest__{batch_info['batch_id']}.json"
    )

    if manifest_path.exists():
        raise FileExistsError(f"Refusing to overwrite manifest: {manifest_path}")

    write_json(manifest_path, batch_info)
    batch_info["manifest_path"] = str(manifest_path)

    print("Submitted Anthropic Message Batch:")
    print(json.dumps(batch_info, indent=2))

    return batch_info


round1_batch_info = submit_anthropic_batch(
    requests=round1_requests,
    round_name="round1",
    plan_path=round1_plan_path,
    local_jsonl_path=round1_jsonl_path,
    manifest_dir=DIRS["round1_manifests"],
)

round1_batch_info

Submitted Anthropic Message Batch:
{
  "run_id": "20260518_095240__3837589d",
  "task_set_id": "taskset_b_additional_prompts",
  "round": "round1",
  "provider": "anthropic",
  "model": "claude-sonnet-4-6",
  "message_batch": {
    "id": "msgbatch_01PfZcx4jGFYEufd1QZnSkdK",
    "archived_at": null,
    "cancel_initiated_at": null,
    "created_at": "2026-05-18T13:52:46.245265Z",
    "ended_at": null,
    "expires_at": "2026-05-19T13:52:46.245265Z",
    "processing_status": "in_progress",
    "request_counts": {
      "canceled": 0,
      "errored": 0,
      "expired": 0,
      "processing": 5400,
      "succeeded": 0
    },
    "results_url": null,
    "type": "message_batch"
  },
  "batch_id": "msgbatch_01PfZcx4jGFYEufd1QZnSkdK",
  "processing_status_at_submission": "in_progress",
  "submitted_at_utc": "2026-05-18T13:52:46.467565+00:00",
  "local_jsonl_path": "ai_data/deflect_creativity/anthropic/model_claude-sonnet-4-6/taskset_b_additional_prompts/run_20260518_095240__3837589d/01_rou

{'run_id': '20260518_095240__3837589d',
 'task_set_id': 'taskset_b_additional_prompts',
 'round': 'round1',
 'provider': 'anthropic',
 'model': 'claude-sonnet-4-6',
 'message_batch': {'id': 'msgbatch_01PfZcx4jGFYEufd1QZnSkdK',
  'archived_at': None,
  'cancel_initiated_at': None,
  'created_at': '2026-05-18T13:52:46.245265Z',
  'ended_at': None,
  'expires_at': '2026-05-19T13:52:46.245265Z',
  'processing_status': 'in_progress',
  'request_counts': {'canceled': 0,
   'errored': 0,
   'expired': 0,
   'processing': 5400,
   'succeeded': 0},
  'results_url': None,
  'type': 'message_batch'},
 'batch_id': 'msgbatch_01PfZcx4jGFYEufd1QZnSkdK',
 'processing_status_at_submission': 'in_progress',
 'submitted_at_utc': '2026-05-18T13:52:46.467565+00:00',
 'local_jsonl_path': 'ai_data/deflect_creativity/anthropic/model_claude-sonnet-4-6/taskset_b_additional_prompts/run_20260518_095240__3837589d/01_round1/batch_inputs/taskset_b_additional_prompts__round1__anthropic__claude-sonnet-4-6__20260518_095

In [17]:
def check_anthropic_batch(batch_id: str) -> dict:
    message_batch = client.messages.batches.retrieve(batch_id)
    info = to_jsonable(message_batch)
    print(json.dumps(info, indent=2))
    return info


round1_status = check_anthropic_batch(round1_batch_info["batch_id"])
round1_status

{
  "id": "msgbatch_01PfZcx4jGFYEufd1QZnSkdK",
  "archived_at": null,
  "cancel_initiated_at": null,
  "created_at": "2026-05-18T13:52:46.245265Z",
  "ended_at": "2026-05-18T14:03:58.982047Z",
  "expires_at": "2026-05-19T13:52:46.245265Z",
  "processing_status": "ended",
  "request_counts": {
    "canceled": 0,
    "errored": 1,
    "expired": 0,
    "processing": 0,
    "succeeded": 5399
  },
  "results_url": "https://api.anthropic.com/v1/messages/batches/msgbatch_01PfZcx4jGFYEufd1QZnSkdK/results",
  "type": "message_batch"
}


{'id': 'msgbatch_01PfZcx4jGFYEufd1QZnSkdK',
 'archived_at': None,
 'cancel_initiated_at': None,
 'created_at': '2026-05-18T13:52:46.245265Z',
 'ended_at': '2026-05-18T14:03:58.982047Z',
 'expires_at': '2026-05-19T13:52:46.245265Z',
 'processing_status': 'ended',
 'request_counts': {'canceled': 0,
  'errored': 1,
  'expired': 0,
  'processing': 0,
  'succeeded': 5399},
 'results_url': 'https://api.anthropic.com/v1/messages/batches/msgbatch_01PfZcx4jGFYEufd1QZnSkdK/results',
 'type': 'message_batch'}

In [21]:
def download_anthropic_batch_results(
    batch_id: str,
    raw_output_dir: Path,
    round_name: str,
) -> Optional[Path]:
    message_batch = client.messages.batches.retrieve(batch_id)
    batch_dump = to_jsonable(message_batch)

    if batch_dump.get("processing_status") != "ended":
        print(f"Batch is not ended yet. Current status: {batch_dump.get('processing_status')}")
        return None

    output_path = raw_output_dir / f"{TASK_SET_ID}__{round_name}__{batch_id}__results.jsonl"

    if output_path.exists():
        raise FileExistsError(f"Refusing to overwrite existing output file: {output_path}")

    n = 0
    with open(output_path, "w", encoding="utf-8") as f:
        for result in client.messages.batches.results(batch_id):
            result_dict = to_jsonable(result)
            f.write(json.dumps(result_dict, ensure_ascii=False) + "\n")
            n += 1

    print(f"Downloaded/streamed {n:,} results to: {output_path}")
    return output_path

def extract_text_from_anthropic_message(message: dict) -> str:
    if not isinstance(message, dict):
        return ""

    texts = []
    for block in message.get("content", []) or []:
        if isinstance(block, dict) and block.get("type") == "text":
            texts.append(block.get("text", ""))

    return "\n".join(texts).strip()


def parse_anthropic_batch_output_to_standard_files(
    batch_output_path: Path,
    plan_path: Path,
    parsed_dir: Path,
    round_name: str,
    batch_id: str,
) -> dict:
    plan_df = pd.read_csv(plan_path)
    plan_by_key = {
        row["request_key"]: row.to_dict()
        for _, row in plan_df.iterrows()
    }

    batch_records = read_jsonl(batch_output_path)

    parsed_jsonl_path = parsed_dir / f"{TASK_SET_ID}__{round_name}__{PROVIDER}__{MODEL_NAME}__{batch_id}__parsed.jsonl"
    parsed_csv_path = parsed_dir / f"{TASK_SET_ID}__{round_name}__{PROVIDER}__{MODEL_NAME}__{batch_id}__parsed.csv"
    parsed_pkl_path = parsed_dir / f"{TASK_SET_ID}__{round_name}__{PROVIDER}__{MODEL_NAME}__{batch_id}__parsed.pkl"

    if parsed_jsonl_path.exists() or parsed_csv_path.exists() or parsed_pkl_path.exists():
        raise FileExistsError("Refusing to overwrite existing parsed files.")

    parsed_records = []
    n_success = 0
    n_empty = 0
    n_error = 0
    n_canceled = 0
    n_expired = 0

    for rec in batch_records:
        request_key = rec.get("custom_id")
        plan_row = plan_by_key.get(request_key, {})
        result = rec.get("result", {})
        result_type = result.get("type")

        if result_type == "succeeded":
            message = result.get("message", {})
            text = clean_model_text(extract_text_from_anthropic_message(message))
            usage = message.get("usage")

            status = "success" if text else "empty_text"
            n_success += int(status == "success")
            n_empty += int(status == "empty_text")

            record = {
                **plan_row,
                "parsed_at_utc": now_iso(),
                "status": status,
                "text": text,
                "provider_response_id": message.get("id"),
                "stop_reason": message.get("stop_reason"),
                "usage": usage,
                "error": None if text else "No text extracted from Anthropic message.",
                "batch_custom_id": request_key,
                "batch_output_file": str(batch_output_path),
                "batch_id": batch_id,
                "raw_result_type": result_type,
            }

        else:
            if result_type == "errored":
                n_error += 1
            elif result_type == "canceled":
                n_canceled += 1
            elif result_type == "expired":
                n_expired += 1
            else:
                n_error += 1

            record = {
                **plan_row,
                "parsed_at_utc": now_iso(),
                "status": result_type or "unknown_error",
                "text": None,
                "provider_response_id": None,
                "stop_reason": None,
                "usage": None,
                "error": result,
                "batch_custom_id": request_key,
                "batch_output_file": str(batch_output_path),
                "batch_id": batch_id,
                "raw_result_type": result_type,
            }

        parsed_records.append(record)
        append_jsonl(parsed_jsonl_path, record)

    parsed_df = pd.DataFrame(parsed_records)
    parsed_df.to_csv(parsed_csv_path, index=False)
    parsed_df.to_pickle(parsed_pkl_path)

    summary = {
        "task_set_id": TASK_SET_ID,
        "round": round_name,
        "batch_id": batch_id,
        "n_records": len(parsed_df),
        "n_success": n_success,
        "n_empty_text": n_empty,
        "n_error": n_error,
        "n_canceled": n_canceled,
        "n_expired": n_expired,
        "parsed_jsonl_path": str(parsed_jsonl_path),
        "parsed_csv_path": str(parsed_csv_path),
        "parsed_pkl_path": str(parsed_pkl_path),
    }

    summary_path = parsed_dir / f"{TASK_SET_ID}__{round_name}__{PROVIDER}__{MODEL_NAME}__{batch_id}__parse_summary.json"
    write_json(summary_path, summary)

    print(json.dumps(summary, indent=2))
    return summary


In [23]:
# Cell A1 — Reuse existing Round 1 raw output file

round1_output_path = (
    DIRS["round1_raw_outputs"]
    / f"{TASK_SET_ID}__round1__{round1_batch_info['batch_id']}__results.jsonl"
)

if not round1_output_path.exists():
    raise FileNotFoundError(f"Could not find existing Round 1 output file: {round1_output_path}")

print("Using existing Round 1 raw output file:")
print(round1_output_path)

Using existing Round 1 raw output file:
ai_data/deflect_creativity/anthropic/model_claude-sonnet-4-6/taskset_b_additional_prompts/run_20260518_095240__3837589d/01_round1/raw_outputs/taskset_b_additional_prompts__round1__msgbatch_01PfZcx4jGFYEufd1QZnSkdK__results.jsonl


In [24]:
# Cell A2 — Load existing parsed Round 1 if available, otherwise parse

existing_round1_parsed_pkls = sorted(
    DIRS["round1_parsed"].glob(
        f"{TASK_SET_ID}__round1__{PROVIDER}__{MODEL_NAME}__{round1_batch_info['batch_id']}__parsed.pkl"
    )
)

if existing_round1_parsed_pkls:
    round1_parsed_pkl_path = existing_round1_parsed_pkls[-1]
    print("Loading existing parsed Round 1 file:")
    print(round1_parsed_pkl_path)
    round1_df = pd.read_pickle(round1_parsed_pkl_path)

else:
    print("No existing parsed Round 1 pickle found. Parsing existing raw output now.")

    round1_parse_summary = parse_anthropic_batch_output_to_standard_files(
        batch_output_path=round1_output_path,
        plan_path=Path(round1_batch_info["plan_path"]),
        parsed_dir=DIRS["round1_parsed"],
        round_name="round1",
        batch_id=round1_batch_info["batch_id"],
    )

    round1_df = pd.read_pickle(round1_parse_summary["parsed_pkl_path"])

print(round1_df.shape)
display(round1_df["status"].value_counts(dropna=False))

display(
    round1_df
    .groupby(["task_id", "strategy", "condition", "status"])
    .size()
    .reset_index(name="n")
    .query("status != 'success'")
)

round1_df.head()

No existing parsed Round 1 pickle found. Parsing existing raw output now.
{
  "task_set_id": "taskset_b_additional_prompts",
  "round": "round1",
  "batch_id": "msgbatch_01PfZcx4jGFYEufd1QZnSkdK",
  "n_records": 5400,
  "n_success": 5399,
  "n_empty_text": 0,
  "n_error": 1,
  "n_canceled": 0,
  "n_expired": 0,
  "parsed_jsonl_path": "ai_data/deflect_creativity/anthropic/model_claude-sonnet-4-6/taskset_b_additional_prompts/run_20260518_095240__3837589d/01_round1/parsed/taskset_b_additional_prompts__round1__anthropic__claude-sonnet-4-6__msgbatch_01PfZcx4jGFYEufd1QZnSkdK__parsed.jsonl",
  "parsed_csv_path": "ai_data/deflect_creativity/anthropic/model_claude-sonnet-4-6/taskset_b_additional_prompts/run_20260518_095240__3837589d/01_round1/parsed/taskset_b_additional_prompts__round1__anthropic__claude-sonnet-4-6__msgbatch_01PfZcx4jGFYEufd1QZnSkdK__parsed.csv",
  "parsed_pkl_path": "ai_data/deflect_creativity/anthropic/model_claude-sonnet-4-6/taskset_b_additional_prompts/run_20260518_095240__

status
success    5399
errored       1
Name: count, dtype: int64

,task_id,strategy,condition,status,n
23,slogan_blood_donation,vanilla,triad,errored,1


,provider,model,task_set_id,round,task_id,task_family,strategy,condition,group_id,agent_id,...,status,text,provider_response_id,stop_reason,usage,error,batch_custom_id,batch_output_file,batch_id,raw_result_type
0,anthropic,claude-sonnet-4-6,taskset_b_additional_prompts,1,story_life_last_seconds,story,vanilla,base,base_069,base_069__a1,...,success,Elara Voss spent a century as a lighthouse kee...,msg_01GC4XiiXevDSR3H6jQVZDDd,end_turn,{'cache_creation': {'ephemeral_1h_input_tokens...,None,r1__dc6fa676bdc5ca5e0df79d42,ai_data/deflect_creativity/anthropic/model_cla...,msgbatch_01PfZcx4jGFYEufd1QZnSkdK,succeeded
1,anthropic,claude-sonnet-4-6,taskset_b_additional_prompts,1,aut_automobile_tire,aut,vanilla,dyad,dyad_060,dyad_060__a2,...,success,A stacked column of old tires filled with soil...,msg_011mibJ597dozVo7igNzxzoq,end_turn,{'cache_creation': {'ephemeral_1h_input_tokens...,None,r1__fa8e9087adece752471741cd,ai_data/deflect_creativity/anthropic/model_cla...,msgbatch_01PfZcx4jGFYEufd1QZnSkdK,succeeded
2,anthropic,claude-sonnet-4-6,taskset_b_additional_prompts,1,aut_automobile_tire,aut,vanilla,dyad,dyad_054,dyad_054__a2,...,success,Stack several automobile tires vertically and ...,msg_01121DSkqCnqWC5kcH2StiFM,end_turn,{'cache_creation': {'ephemeral_1h_input_tokens...,None,r1__97daabc7e3df4cb885d09091,ai_data/deflect_creativity/anthropic/model_cla...,msgbatch_01PfZcx4jGFYEufd1QZnSkdK,succeeded
3,anthropic,claude-sonnet-4-6,taskset_b_additional_prompts,1,aut_automobile_tire,aut,diverge,base,base_001,base_001__a1,...,success,"Stacked and filled with soil, used as a raised...",msg_01V8hmvuoGa7yMmjir56tHJC,end_turn,{'cache_creation': {'ephemeral_1h_input_tokens...,None,r1__ed4b70c212cc324219b1d60f,ai_data/deflect_creativity/anthropic/model_cla...,msgbatch_01PfZcx4jGFYEufd1QZnSkdK,succeeded
4,anthropic,claude-sonnet-4-6,taskset_b_additional_prompts,1,aut_wooden_pencil,aut,vanilla,base,base_101,base_101__a1,...,success,Use the pencil as a splint to stabilize a brok...,msg_016NSvonWQg7Jmmu4jvzj3EJ,end_turn,{'cache_creation': {'ephemeral_1h_input_tokens...,None,r1__93dde72ded080d87b41f166d,ai_data/deflect_creativity/anthropic/model_cla...,msgbatch_01PfZcx4jGFYEufd1QZnSkdK,succeeded


In [25]:
# Cell B — Identify failed or empty Round 1 records

round1_bad_df = round1_df[
    ~round1_df["status"].eq("success")
    | round1_df["text"].isna()
    | round1_df["text"].astype(str).str.strip().eq("")
].copy()

print("Bad Round 1 records:", len(round1_bad_df))

display(
    round1_bad_df[[
        "request_key",
        "task_id",
        "task_family",
        "strategy",
        "condition",
        "group_id",
        "agent_id",
        "status",
        "error",
    ]]
)

if len(round1_bad_df) == 0:
    print("No repair needed. Proceed to Round 2 plan-building.")
elif len(round1_bad_df) > 10:
    raise RuntimeError("Unexpectedly many failed records. Inspect before repairing.")

Bad Round 1 records: 1


,request_key,task_id,task_family,strategy,condition,group_id,agent_id,status,error
500,r1__b70fb9ab430313477cf4d1df,slogan_blood_donation,slogan,vanilla,triad,triad_010,triad_010__a2,errored,{'error': {'error': {'message': 'API key valid...


In [26]:
# Cell C — Submit Round 1 repair batch for failed/empty records

def build_anthropic_repair_requests_from_bad_records(
    bad_df: pd.DataFrame,
    original_plan_path: Path,
) -> tuple[list[dict], pd.DataFrame]:
    original_plan_df = pd.read_csv(original_plan_path)

    failed_keys = bad_df["request_key"].dropna().astype(str).tolist()

    repair_plan_df = original_plan_df[
        original_plan_df["request_key"].astype(str).isin(failed_keys)
    ].copy()

    if len(repair_plan_df) != len(failed_keys):
        print("Failed keys:", failed_keys)
        print("Matched repair-plan rows:", len(repair_plan_df))
        raise RuntimeError("Could not match every failed request_key to the original plan.")

    repair_plan_df["original_request_key"] = repair_plan_df["request_key"]

    repair_plan_df["request_key"] = repair_plan_df["request_key"].map(
        lambda x: "repair_" + str(x)
    )

    repair_plan_df["repair_of_batch_id"] = round1_batch_info["batch_id"]
    repair_plan_df["repair_created_at_utc"] = now_iso()

    repair_requests = []

    for _, row in repair_plan_df.iterrows():
        repair_requests.append({
            "custom_id": row["request_key"],
            "params": make_anthropic_message_params(row),
        })

    return repair_requests, repair_plan_df


if len(round1_bad_df) > 0:
    round1_repair_requests, round1_repair_plan_df = build_anthropic_repair_requests_from_bad_records(
        bad_df=round1_bad_df,
        original_plan_path=Path(round1_batch_info["plan_path"]),
    )

    timestamp = dt.datetime.now().strftime("%Y%m%d_%H%M%S")

    round1_repair_plan_path = (
        DIRS["round1_plans"]
        / f"{TASK_SET_ID}__round1_repair__{PROVIDER}__{MODEL_NAME}__{timestamp}__plan.csv"
    )

    round1_repair_jsonl_path = (
        DIRS["round1_batch_inputs"]
        / f"{TASK_SET_ID}__round1_repair__{PROVIDER}__{MODEL_NAME}__{timestamp}__local_batch_requests.jsonl"
    )

    if round1_repair_plan_path.exists() or round1_repair_jsonl_path.exists():
        raise FileExistsError("Refusing to overwrite repair files.")

    round1_repair_plan_df.to_csv(round1_repair_plan_path, index=False)

    with open(round1_repair_jsonl_path, "w", encoding="utf-8") as f:
        for req in round1_repair_requests:
            f.write(json.dumps(req, ensure_ascii=False) + "\n")

    round1_repair_batch_info = submit_anthropic_batch(
        requests=round1_repair_requests,
        round_name="round1_repair",
        plan_path=round1_repair_plan_path,
        local_jsonl_path=round1_repair_jsonl_path,
        manifest_dir=DIRS["round1_manifests"],
    )

    display(round1_repair_plan_df)
    display(round1_repair_batch_info)

else:
    round1_repair_batch_info = None
    print("No repair batch submitted.")

Submitted Anthropic Message Batch:
{
  "run_id": "20260518_095240__3837589d",
  "task_set_id": "taskset_b_additional_prompts",
  "round": "round1_repair",
  "provider": "anthropic",
  "model": "claude-sonnet-4-6",
  "message_batch": {
    "id": "msgbatch_01TSZejqgf6nMx4VdzwdvYj7",
    "archived_at": null,
    "cancel_initiated_at": null,
    "created_at": "2026-05-18T14:38:57.121512Z",
    "ended_at": null,
    "expires_at": "2026-05-19T14:38:57.121512Z",
    "processing_status": "in_progress",
    "request_counts": {
      "canceled": 0,
      "errored": 0,
      "expired": 0,
      "processing": 1,
      "succeeded": 0
    },
    "results_url": null,
    "type": "message_batch"
  },
  "batch_id": "msgbatch_01TSZejqgf6nMx4VdzwdvYj7",
  "processing_status_at_submission": "in_progress",
  "submitted_at_utc": "2026-05-18T14:38:57.180920+00:00",
  "local_jsonl_path": "ai_data/deflect_creativity/anthropic/model_claude-sonnet-4-6/taskset_b_additional_prompts/run_20260518_095240__3837589d/01

,provider,model,task_set_id,round,task_id,task_family,strategy,condition,group_id,agent_id,agent_index,request_key,system_instructions,user_prompt,temperature,max_output_tokens,created_at_utc,original_request_key,repair_of_batch_id,repair_created_at_utc
328,anthropic,claude-sonnet-4-6,taskset_b_additional_prompts,1,slogan_blood_donation,slogan,vanilla,triad,triad_010,triad_010__a2,2,repair_r1__b70fb9ab430313477cf4d1df,You are participating in a controlled creativi...,You are part of the communications team at a n...,1.0,60,2026-05-18T13:52:42.898409+00:00,r1__b70fb9ab430313477cf4d1df,msgbatch_01PfZcx4jGFYEufd1QZnSkdK,2026-05-18T14:38:56.669594+00:00


{'run_id': '20260518_095240__3837589d',
 'task_set_id': 'taskset_b_additional_prompts',
 'round': 'round1_repair',
 'provider': 'anthropic',
 'model': 'claude-sonnet-4-6',
 'message_batch': {'id': 'msgbatch_01TSZejqgf6nMx4VdzwdvYj7',
  'archived_at': None,
  'cancel_initiated_at': None,
  'created_at': '2026-05-18T14:38:57.121512Z',
  'ended_at': None,
  'expires_at': '2026-05-19T14:38:57.121512Z',
  'processing_status': 'in_progress',
  'request_counts': {'canceled': 0,
   'errored': 0,
   'expired': 0,
   'processing': 1,
   'succeeded': 0},
  'results_url': None,
  'type': 'message_batch'},
 'batch_id': 'msgbatch_01TSZejqgf6nMx4VdzwdvYj7',
 'processing_status_at_submission': 'in_progress',
 'submitted_at_utc': '2026-05-18T14:38:57.180920+00:00',
 'local_jsonl_path': 'ai_data/deflect_creativity/anthropic/model_claude-sonnet-4-6/taskset_b_additional_prompts/run_20260518_095240__3837589d/01_round1/batch_inputs/taskset_b_additional_prompts__round1_repair__anthropic__claude-sonnet-4-6__2

In [29]:
# Cell D — Check Round 1 repair batch status

if round1_repair_batch_info is not None:
    round1_repair_status = check_anthropic_batch(round1_repair_batch_info["batch_id"])
    display(round1_repair_status)
else:
    print("No repair batch exists.")

{
  "id": "msgbatch_01TSZejqgf6nMx4VdzwdvYj7",
  "archived_at": null,
  "cancel_initiated_at": null,
  "created_at": "2026-05-18T14:38:57.121512Z",
  "ended_at": "2026-05-18T14:41:05.901715Z",
  "expires_at": "2026-05-19T14:38:57.121512Z",
  "processing_status": "ended",
  "request_counts": {
    "canceled": 0,
    "errored": 0,
    "expired": 0,
    "processing": 0,
    "succeeded": 1
  },
  "results_url": "https://api.anthropic.com/v1/messages/batches/msgbatch_01TSZejqgf6nMx4VdzwdvYj7/results",
  "type": "message_batch"
}


{'id': 'msgbatch_01TSZejqgf6nMx4VdzwdvYj7',
 'archived_at': None,
 'cancel_initiated_at': None,
 'created_at': '2026-05-18T14:38:57.121512Z',
 'ended_at': '2026-05-18T14:41:05.901715Z',
 'expires_at': '2026-05-19T14:38:57.121512Z',
 'processing_status': 'ended',
 'request_counts': {'canceled': 0,
  'errored': 0,
  'expired': 0,
  'processing': 0,
  'succeeded': 1},
 'results_url': 'https://api.anthropic.com/v1/messages/batches/msgbatch_01TSZejqgf6nMx4VdzwdvYj7/results',
 'type': 'message_batch'}

In [30]:
# Cell E — Download/reuse and parse Round 1 repair batch

if round1_repair_batch_info is not None:
    round1_repair_output_path = (
        DIRS["round1_raw_outputs"]
        / f"{TASK_SET_ID}__round1_repair__{round1_repair_batch_info['batch_id']}__results.jsonl"
    )

    if round1_repair_output_path.exists():
        print("Using existing repair raw output file:")
        print(round1_repair_output_path)
    else:
        round1_repair_output_path = download_anthropic_batch_results(
            batch_id=round1_repair_batch_info["batch_id"],
            raw_output_dir=DIRS["round1_raw_outputs"],
            round_name="round1_repair",
        )

    existing_repair_parsed_pkls = sorted(
        DIRS["round1_parsed"].glob(
            f"{TASK_SET_ID}__round1_repair__{PROVIDER}__{MODEL_NAME}__{round1_repair_batch_info['batch_id']}__parsed.pkl"
        )
    )

    if existing_repair_parsed_pkls:
        round1_repair_pkl_path = existing_repair_parsed_pkls[-1]
        print("Loading existing parsed repair file:")
        print(round1_repair_pkl_path)
        round1_repair_df = pd.read_pickle(round1_repair_pkl_path)
    else:
        round1_repair_parse_summary = parse_anthropic_batch_output_to_standard_files(
            batch_output_path=round1_repair_output_path,
            plan_path=Path(round1_repair_batch_info["plan_path"]),
            parsed_dir=DIRS["round1_parsed"],
            round_name="round1_repair",
            batch_id=round1_repair_batch_info["batch_id"],
        )

        round1_repair_df = pd.read_pickle(round1_repair_parse_summary["parsed_pkl_path"])

    print(round1_repair_df.shape)
    display(round1_repair_df["status"].value_counts(dropna=False))

    display(round1_repair_df[[
        "request_key",
        "original_request_key",
        "task_id",
        "strategy",
        "condition",
        "group_id",
        "agent_id",
        "status",
        "text",
        "error",
    ]])

else:
    round1_repair_df = pd.DataFrame()
    print("No repair batch to parse.")

Downloaded/streamed 1 results to: ai_data/deflect_creativity/anthropic/model_claude-sonnet-4-6/taskset_b_additional_prompts/run_20260518_095240__3837589d/01_round1/raw_outputs/taskset_b_additional_prompts__round1_repair__msgbatch_01TSZejqgf6nMx4VdzwdvYj7__results.jsonl
{
  "task_set_id": "taskset_b_additional_prompts",
  "round": "round1_repair",
  "batch_id": "msgbatch_01TSZejqgf6nMx4VdzwdvYj7",
  "n_records": 1,
  "n_success": 1,
  "n_empty_text": 0,
  "n_error": 0,
  "n_canceled": 0,
  "n_expired": 0,
  "parsed_jsonl_path": "ai_data/deflect_creativity/anthropic/model_claude-sonnet-4-6/taskset_b_additional_prompts/run_20260518_095240__3837589d/01_round1/parsed/taskset_b_additional_prompts__round1_repair__anthropic__claude-sonnet-4-6__msgbatch_01TSZejqgf6nMx4VdzwdvYj7__parsed.jsonl",
  "parsed_csv_path": "ai_data/deflect_creativity/anthropic/model_claude-sonnet-4-6/taskset_b_additional_prompts/run_20260518_095240__3837589d/01_round1/parsed/taskset_b_additional_prompts__round1_repair__

status
success    1
Name: count, dtype: int64

,request_key,original_request_key,task_id,strategy,condition,group_id,agent_id,status,text,error
0,repair_r1__b70fb9ab430313477cf4d1df,r1__b70fb9ab430313477cf4d1df,slogan_blood_donation,vanilla,triad,triad_010,triad_010__a2,success,Your blood runs in their veins.,None


In [31]:
# Cell F — Merge repaired Round 1 records and save repaired Round 1 file

def merge_round1_with_repairs(
    original_df: pd.DataFrame,
    repair_df: pd.DataFrame,
) -> pd.DataFrame:
    if repair_df.empty:
        return original_df.copy()

    repair_success = repair_df[repair_df["status"].eq("success")].copy()

    if len(repair_success) != len(repair_df):
        display(repair_df[repair_df["status"] != "success"])
        raise RuntimeError("At least one repair request failed. Inspect before proceeding.")

    if "original_request_key" not in repair_success.columns:
        raise RuntimeError("Repair dataframe is missing original_request_key.")

    repaired_df = original_df.copy()

    for _, repaired_row in repair_success.iterrows():
        original_key = repaired_row["original_request_key"]

        mask = repaired_df["request_key"].astype(str).eq(str(original_key))

        if mask.sum() != 1:
            raise RuntimeError(f"Could not uniquely locate original failed row: {original_key}")

        replacement = repaired_row.copy()
        replacement["repair_request_key"] = repaired_row["request_key"]
        replacement["request_key"] = original_key
        replacement["repaired_from_batch_id"] = repaired_row.get("batch_id")
        replacement["repair_applied_at_utc"] = now_iso()

        for col in replacement.index:
            if col not in repaired_df.columns:
                repaired_df[col] = None

        repaired_df.loc[mask, replacement.index] = replacement.values

    return repaired_df


round1_df_repaired = merge_round1_with_repairs(
    original_df=round1_df,
    repair_df=round1_repair_df,
)

print(round1_df_repaired.shape)
display(round1_df_repaired["status"].value_counts(dropna=False))

round1_remaining_bad = round1_df_repaired[
    ~round1_df_repaired["status"].eq("success")
    | round1_df_repaired["text"].isna()
    | round1_df_repaired["text"].astype(str).str.strip().eq("")
].copy()

print("Remaining bad Round 1 records:", len(round1_remaining_bad))

display(round1_remaining_bad[[
    "request_key",
    "task_id",
    "strategy",
    "condition",
    "group_id",
    "agent_id",
    "status",
    "error",
]])

expected_round1 = len(TASK_SETTINGS) * len(STRATEGIES) * (N_BASE_AGENTS + 2 * N_DYADS + 3 * N_TRIADS)

print("Expected Round 1 rows:", expected_round1)
print("Actual Round 1 rows:  ", len(round1_df_repaired))

if len(round1_df_repaired) != expected_round1:
    raise RuntimeError("Round 1 repaired row count does not match expectation.")

if len(round1_remaining_bad) > 0:
    raise RuntimeError("Round 1 still has failed or empty records after repair.")

timestamp = dt.datetime.now().strftime("%Y%m%d_%H%M%S")

round1_repaired_csv_path = (
    DIRS["round1_parsed"]
    / f"{TASK_SET_ID}__round1__{PROVIDER}__{MODEL_NAME}__repaired__{timestamp}.csv"
)

round1_repaired_pkl_path = (
    DIRS["round1_parsed"]
    / f"{TASK_SET_ID}__round1__{PROVIDER}__{MODEL_NAME}__repaired__{timestamp}.pkl"
)

if round1_repaired_csv_path.exists() or round1_repaired_pkl_path.exists():
    raise FileExistsError("Refusing to overwrite repaired Round 1 files.")

round1_df_repaired.to_csv(round1_repaired_csv_path, index=False)
round1_df_repaired.to_pickle(round1_repaired_pkl_path)

print("Saved repaired Round 1 files:")
print(round1_repaired_csv_path)
print(round1_repaired_pkl_path)

round1_df = round1_df_repaired

(5400, 34)


status
success    5400
Name: count, dtype: int64

Remaining bad Round 1 records: 0


,request_key,task_id,strategy,condition,group_id,agent_id,status,error


Expected Round 1 rows: 5400
Actual Round 1 rows:   5400
Saved repaired Round 1 files:
ai_data/deflect_creativity/anthropic/model_claude-sonnet-4-6/taskset_b_additional_prompts/run_20260518_095240__3837589d/01_round1/parsed/taskset_b_additional_prompts__round1__anthropic__claude-sonnet-4-6__repaired__20260518_104203.csv
ai_data/deflect_creativity/anthropic/model_claude-sonnet-4-6/taskset_b_additional_prompts/run_20260518_095240__3837589d/01_round1/parsed/taskset_b_additional_prompts__round1__anthropic__claude-sonnet-4-6__repaired__20260518_104203.pkl


In [ ]:
# Optional reload after kernel restart:
# manifest_path = Path("ai_data/deflect_creativity/anthropic/model_claude-sonnet-4-6/taskset_b_additional_prompts/run_.../01_round1/manifests/...")
# round1_batch_info = read_json(manifest_path)
# DATA_ROOT = Path(round1_batch_info["data_root"])
# round1_batch_info

In [ ]:
# round1_output_path = download_anthropic_batch_results(
#     batch_id=round1_batch_info["batch_id"],
#     raw_output_dir=DIRS["round1_raw_outputs"],
#     round_name="round1",
# )

# round1_output_path

In [ ]:
# round1_parse_summary = parse_anthropic_batch_output_to_standard_files(
#     batch_output_path=round1_output_path,
#     plan_path=Path(round1_batch_info["plan_path"]),
#     parsed_dir=DIRS["round1_parsed"],
#     round_name="round1",
#     batch_id=round1_batch_info["batch_id"],
# )

# round1_df = pd.read_pickle(round1_parse_summary["parsed_pkl_path"])
# print(round1_df.shape)
# display(round1_df["status"].value_counts(dropna=False))
# round1_df.head()

In [32]:
expected_round1 = len(TASK_SETTINGS) * len(STRATEGIES) * len(agents_df)
actual_round1 = len(round1_df)

print("Expected Round 1 rows:", expected_round1)
print("Actual Round 1 rows:  ", actual_round1)

display(round1_df.groupby(["task_id", "strategy", "condition", "status"]).size().reset_index(name="n"))

if actual_round1 != expected_round1:
    print("WARNING: Row count mismatch. Inspect before proceeding.")

if (round1_df["status"] != "success").any():
    print("WARNING: Some Round 1 calls failed or returned empty text. Inspect before proceeding to Round 2.")
    display(round1_df[round1_df["status"] != "success"].head(20))
else:
    print("Round 1 looks complete.")

Expected Round 1 rows: 5400
Actual Round 1 rows:   5400


,task_id,strategy,condition,status,n
0,aut_automobile_tire,diverge,base,success,150
1,aut_automobile_tire,diverge,dyad,success,150
2,aut_automobile_tire,diverge,triad,success,150
3,aut_automobile_tire,vanilla,base,success,150
4,aut_automobile_tire,vanilla,dyad,success,150
5,aut_automobile_tire,vanilla,triad,success,150
6,aut_key,diverge,base,success,150
7,aut_key,diverge,dyad,success,150
8,aut_key,diverge,triad,success,150
9,aut_key,vanilla,base,success,150


Round 1 looks complete.


In [33]:
def get_task_by_id(task_id: str) -> dict:
    for t in TASK_SETTINGS:
        if t["task_id"] == task_id:
            return t
    raise KeyError(task_id)


def build_round2_plan(round1_df: pd.DataFrame) -> pd.DataFrame:
    rows = []
    r1_success = round1_df[round1_df["status"] == "success"].copy()

    required_cols = ["task_id", "strategy", "condition", "group_id", "agent_id"]
    if r1_success.duplicated(required_cols).any():
        dupes = r1_success[r1_success.duplicated(required_cols, keep=False)].sort_values(required_cols)
        raise ValueError(f"Duplicate Round 1 successful records:\n{dupes[required_cols + ['text']].head()}")

    for (task_id, strategy, condition, group_id), group in r1_success.groupby(
        ["task_id", "strategy", "condition", "group_id"],
        sort=True,
    ):
        task = get_task_by_id(task_id)
        group = group.sort_values("agent_index").copy()

        expected_group_size = {"base": 1, "dyad": 2, "triad": 3}[condition]
        if len(group) != expected_group_size:
            raise ValueError(
                f"Group size mismatch for {(task_id, strategy, condition, group_id)}: "
                f"expected {expected_group_size}, got {len(group)}"
            )

        for _, ego in group.iterrows():
            peer_rows = group[group["agent_id"] != ego["agent_id"]].sort_values("agent_index")
            peer_texts = peer_rows["text"].tolist()
            peer_agent_ids = peer_rows["agent_id"].tolist()

            user_prompt = build_round2_prompt(
                task=task,
                strategy=strategy,
                condition=condition,
                self_round1=ego["text"],
                peer_round1_texts=peer_texts,
            )

            request_basis = {
                "provider": PROVIDER,
                "model": MODEL_NAME,
                "task_set_id": TASK_SET_ID,
                "round": 2,
                "task_id": task_id,
                "task_family": task["task_family"],
                "strategy": strategy,
                "condition": condition,
                "group_id": group_id,
                "agent_id": ego["agent_id"],
                "agent_index": int(ego["agent_index"]),
                "self_round1_request_key": ego["request_key"],
                "peer_round1_agent_ids": "|".join(peer_agent_ids),
            }

            request_key = "r2__" + stable_hash(json.dumps(request_basis, sort_keys=True), 24)

            rows.append({
                **request_basis,
                "request_key": request_key,
                "system_instructions": SYSTEM_INSTRUCTIONS,
                "user_prompt": user_prompt,
                "temperature": TEMPERATURE,
                "max_output_tokens": MAX_OUTPUT_TOKENS_BY_FAMILY[task["task_family"]],
                "self_round1_text": ego["text"],
                "peer_round1_texts_json": json.dumps(peer_texts, ensure_ascii=False),
                "created_at_utc": now_iso(),
            })

    plan_df = pd.DataFrame(rows)

    if plan_df["request_key"].duplicated().any():
        dupes = plan_df[plan_df["request_key"].duplicated(keep=False)].sort_values("request_key")
        raise ValueError(f"Duplicate request_key detected:\n{dupes.head()}")

    return plan_df


round2_plan_df = build_round2_plan(round1_df)

expected_round2 = expected_round1

print("Expected Round 2 requests:", expected_round2)
print("Actual Round 2 requests:  ", len(round2_plan_df))

display(round2_plan_df.groupby(["task_id", "strategy", "condition"]).size().reset_index(name="n"))
round2_plan_df.head()

Expected Round 2 requests: 5400
Actual Round 2 requests:   5400


,task_id,strategy,condition,n
0,aut_automobile_tire,diverge,base,150
1,aut_automobile_tire,diverge,dyad,150
2,aut_automobile_tire,diverge,triad,150
3,aut_automobile_tire,vanilla,base,150
4,aut_automobile_tire,vanilla,dyad,150
5,aut_automobile_tire,vanilla,triad,150
6,aut_key,diverge,base,150
7,aut_key,diverge,dyad,150
8,aut_key,diverge,triad,150
9,aut_key,vanilla,base,150


,provider,model,task_set_id,round,task_id,task_family,strategy,condition,group_id,agent_id,...,self_round1_request_key,peer_round1_agent_ids,request_key,system_instructions,user_prompt,temperature,max_output_tokens,self_round1_text,peer_round1_texts_json,created_at_utc
0,anthropic,claude-sonnet-4-6,taskset_b_additional_prompts,2,aut_automobile_tire,aut,diverge,base,base_001,base_001__a1,...,r1__ed4b70c212cc324219b1d60f,,r2__668c1941d0a0da8f68ddd2f2,You are participating in a controlled creativi...,You are participating in a creativity task.\n\...,1.0,120,"Stacked and filled with soil, used as a raised...",[],2026-05-18T14:42:16.990395+00:00
1,anthropic,claude-sonnet-4-6,taskset_b_additional_prompts,2,aut_automobile_tire,aut,diverge,base,base_002,base_002__a1,...,r1__943c81b2745feadd05d09e9f,,r2__8a94adc496b363405ff863ba,You are participating in a controlled creativi...,You are participating in a creativity task.\n\...,1.0,120,Strung horizontally between two trees with rop...,[],2026-05-18T14:42:16.991013+00:00
2,anthropic,claude-sonnet-4-6,taskset_b_additional_prompts,2,aut_automobile_tire,aut,diverge,base,base_003,base_003__a1,...,r1__47e676e0a2df1bfb1e9d9183,,r2__22f0b547a3160fe3322da286,You are participating in a controlled creativi...,You are participating in a creativity task.\n\...,1.0,120,Suspend a tire vertically from a sturdy tree b...,[],2026-05-18T14:42:16.991541+00:00
3,anthropic,claude-sonnet-4-6,taskset_b_additional_prompts,2,aut_automobile_tire,aut,diverge,base,base_004,base_004__a1,...,r1__7b6deb386a9b3a84c8ddcbd1,,r2__58f14167e3d1a442ec1b5942,You are participating in a controlled creativi...,You are participating in a creativity task.\n\...,1.0,120,Strung horizontally between two trees with a p...,[],2026-05-18T14:42:16.992011+00:00
4,anthropic,claude-sonnet-4-6,taskset_b_additional_prompts,2,aut_automobile_tire,aut,diverge,base,base_005,base_005__a1,...,r1__c4f867e1d900be5898fbf715,,r2__783bd644bd15acd2312e9ea8,You are participating in a controlled creativi...,You are participating in a creativity task.\n\...,1.0,120,Stack several automobile tires vertically and ...,[],2026-05-18T14:42:16.992460+00:00


In [34]:
round2_requests, round2_jsonl_path, round2_plan_path = make_anthropic_batch_request_files(
    plan_df=round2_plan_df,
    round_name="round2",
    batch_input_dir=DIRS["round2_batch_inputs"],
)

round2_jsonl_path, round2_plan_path

Wrote plan:          ai_data/deflect_creativity/anthropic/model_claude-sonnet-4-6/taskset_b_additional_prompts/run_20260518_095240__3837589d/02_round2/plans/taskset_b_additional_prompts__round2__anthropic__claude-sonnet-4-6__20260518_104222__plan.csv
Wrote local JSONL:   ai_data/deflect_creativity/anthropic/model_claude-sonnet-4-6/taskset_b_additional_prompts/run_20260518_095240__3837589d/02_round2/batch_inputs/taskset_b_additional_prompts__round2__anthropic__claude-sonnet-4-6__20260518_104222__local_batch_requests.jsonl
Batch request count: 5,400


(PosixPath('ai_data/deflect_creativity/anthropic/model_claude-sonnet-4-6/taskset_b_additional_prompts/run_20260518_095240__3837589d/02_round2/batch_inputs/taskset_b_additional_prompts__round2__anthropic__claude-sonnet-4-6__20260518_104222__local_batch_requests.jsonl'),
 PosixPath('ai_data/deflect_creativity/anthropic/model_claude-sonnet-4-6/taskset_b_additional_prompts/run_20260518_095240__3837589d/02_round2/plans/taskset_b_additional_prompts__round2__anthropic__claude-sonnet-4-6__20260518_104222__plan.csv'))

In [35]:
round2_batch_info = submit_anthropic_batch(
    requests=round2_requests,
    round_name="round2",
    plan_path=round2_plan_path,
    local_jsonl_path=round2_jsonl_path,
    manifest_dir=DIRS["round2_manifests"],
)

round2_batch_info

Submitted Anthropic Message Batch:
{
  "run_id": "20260518_095240__3837589d",
  "task_set_id": "taskset_b_additional_prompts",
  "round": "round2",
  "provider": "anthropic",
  "model": "claude-sonnet-4-6",
  "message_batch": {
    "id": "msgbatch_01BAMDhQnbTbSnePARZesaMM",
    "archived_at": null,
    "cancel_initiated_at": null,
    "created_at": "2026-05-18T14:42:27.371347Z",
    "ended_at": null,
    "expires_at": "2026-05-19T14:42:27.371347Z",
    "processing_status": "in_progress",
    "request_counts": {
      "canceled": 0,
      "errored": 0,
      "expired": 0,
      "processing": 5400,
      "succeeded": 0
    },
    "results_url": null,
    "type": "message_batch"
  },
  "batch_id": "msgbatch_01BAMDhQnbTbSnePARZesaMM",
  "processing_status_at_submission": "in_progress",
  "submitted_at_utc": "2026-05-18T14:42:27.713197+00:00",
  "local_jsonl_path": "ai_data/deflect_creativity/anthropic/model_claude-sonnet-4-6/taskset_b_additional_prompts/run_20260518_095240__3837589d/02_rou

{'run_id': '20260518_095240__3837589d',
 'task_set_id': 'taskset_b_additional_prompts',
 'round': 'round2',
 'provider': 'anthropic',
 'model': 'claude-sonnet-4-6',
 'message_batch': {'id': 'msgbatch_01BAMDhQnbTbSnePARZesaMM',
  'archived_at': None,
  'cancel_initiated_at': None,
  'created_at': '2026-05-18T14:42:27.371347Z',
  'ended_at': None,
  'expires_at': '2026-05-19T14:42:27.371347Z',
  'processing_status': 'in_progress',
  'request_counts': {'canceled': 0,
   'errored': 0,
   'expired': 0,
   'processing': 5400,
   'succeeded': 0},
  'results_url': None,
  'type': 'message_batch'},
 'batch_id': 'msgbatch_01BAMDhQnbTbSnePARZesaMM',
 'processing_status_at_submission': 'in_progress',
 'submitted_at_utc': '2026-05-18T14:42:27.713197+00:00',
 'local_jsonl_path': 'ai_data/deflect_creativity/anthropic/model_claude-sonnet-4-6/taskset_b_additional_prompts/run_20260518_095240__3837589d/02_round2/batch_inputs/taskset_b_additional_prompts__round2__anthropic__claude-sonnet-4-6__20260518_104

In [46]:
round2_status = check_anthropic_batch(round2_batch_info["batch_id"])
round2_status

{
  "id": "msgbatch_01BAMDhQnbTbSnePARZesaMM",
  "archived_at": null,
  "cancel_initiated_at": null,
  "created_at": "2026-05-18T14:42:27.371347Z",
  "ended_at": "2026-05-18T14:57:59.548215Z",
  "expires_at": "2026-05-19T14:42:27.371347Z",
  "processing_status": "ended",
  "request_counts": {
    "canceled": 0,
    "errored": 2,
    "expired": 0,
    "processing": 0,
    "succeeded": 5398
  },
  "results_url": "https://api.anthropic.com/v1/messages/batches/msgbatch_01BAMDhQnbTbSnePARZesaMM/results",
  "type": "message_batch"
}


{'id': 'msgbatch_01BAMDhQnbTbSnePARZesaMM',
 'archived_at': None,
 'cancel_initiated_at': None,
 'created_at': '2026-05-18T14:42:27.371347Z',
 'ended_at': '2026-05-18T14:57:59.548215Z',
 'expires_at': '2026-05-19T14:42:27.371347Z',
 'processing_status': 'ended',
 'request_counts': {'canceled': 0,
  'errored': 2,
  'expired': 0,
  'processing': 0,
  'succeeded': 5398},
 'results_url': 'https://api.anthropic.com/v1/messages/batches/msgbatch_01BAMDhQnbTbSnePARZesaMM/results',
 'type': 'message_batch'}

In [47]:
# Cell R2-A — Reuse or download Round 2 raw output

round2_output_path = (
    DIRS["round2_raw_outputs"]
    / f"{TASK_SET_ID}__round2__{round2_batch_info['batch_id']}__results.jsonl"
)

if round2_output_path.exists():
    print("Using existing Round 2 raw output file:")
    print(round2_output_path)
else:
    round2_output_path = download_anthropic_batch_results(
        batch_id=round2_batch_info["batch_id"],
        raw_output_dir=DIRS["round2_raw_outputs"],
        round_name="round2",
    )

round2_output_path

Downloaded/streamed 5,400 results to: ai_data/deflect_creativity/anthropic/model_claude-sonnet-4-6/taskset_b_additional_prompts/run_20260518_095240__3837589d/02_round2/raw_outputs/taskset_b_additional_prompts__round2__msgbatch_01BAMDhQnbTbSnePARZesaMM__results.jsonl


PosixPath('ai_data/deflect_creativity/anthropic/model_claude-sonnet-4-6/taskset_b_additional_prompts/run_20260518_095240__3837589d/02_round2/raw_outputs/taskset_b_additional_prompts__round2__msgbatch_01BAMDhQnbTbSnePARZesaMM__results.jsonl')

In [48]:
# Cell R2-B — Load existing parsed Round 2 if available, otherwise parse

existing_round2_parsed_pkls = sorted(
    DIRS["round2_parsed"].glob(
        f"{TASK_SET_ID}__round2__{PROVIDER}__{MODEL_NAME}__{round2_batch_info['batch_id']}__parsed.pkl"
    )
)

if existing_round2_parsed_pkls:
    round2_parsed_pkl_path = existing_round2_parsed_pkls[-1]
    print("Loading existing parsed Round 2 file:")
    print(round2_parsed_pkl_path)
    round2_df = pd.read_pickle(round2_parsed_pkl_path)

else:
    print("No existing parsed Round 2 pickle found. Parsing existing raw output now.")

    round2_parse_summary = parse_anthropic_batch_output_to_standard_files(
        batch_output_path=round2_output_path,
        plan_path=Path(round2_batch_info["plan_path"]),
        parsed_dir=DIRS["round2_parsed"],
        round_name="round2",
        batch_id=round2_batch_info["batch_id"],
    )

    round2_df = pd.read_pickle(round2_parse_summary["parsed_pkl_path"])

print(round2_df.shape)
display(round2_df["status"].value_counts(dropna=False))

display(
    round2_df
    .groupby(["task_id", "strategy", "condition", "status"])
    .size()
    .reset_index(name="n")
    .query("status != 'success'")
)

round2_df.head()

No existing parsed Round 2 pickle found. Parsing existing raw output now.
{
  "task_set_id": "taskset_b_additional_prompts",
  "round": "round2",
  "batch_id": "msgbatch_01BAMDhQnbTbSnePARZesaMM",
  "n_records": 5400,
  "n_success": 5398,
  "n_empty_text": 0,
  "n_error": 2,
  "n_canceled": 0,
  "n_expired": 0,
  "parsed_jsonl_path": "ai_data/deflect_creativity/anthropic/model_claude-sonnet-4-6/taskset_b_additional_prompts/run_20260518_095240__3837589d/02_round2/parsed/taskset_b_additional_prompts__round2__anthropic__claude-sonnet-4-6__msgbatch_01BAMDhQnbTbSnePARZesaMM__parsed.jsonl",
  "parsed_csv_path": "ai_data/deflect_creativity/anthropic/model_claude-sonnet-4-6/taskset_b_additional_prompts/run_20260518_095240__3837589d/02_round2/parsed/taskset_b_additional_prompts__round2__anthropic__claude-sonnet-4-6__msgbatch_01BAMDhQnbTbSnePARZesaMM__parsed.csv",
  "parsed_pkl_path": "ai_data/deflect_creativity/anthropic/model_claude-sonnet-4-6/taskset_b_additional_prompts/run_20260518_095240__

status
success    5398
errored       2
Name: count, dtype: int64

,task_id,strategy,condition,status,n
4,aut_automobile_tire,vanilla,dyad,errored,1
32,story_life_last_seconds,diverge,dyad,errored,1


,provider,model,task_set_id,round,task_id,task_family,strategy,condition,group_id,agent_id,...,status,text,provider_response_id,stop_reason,usage,error,batch_custom_id,batch_output_file,batch_id,raw_result_type
0,anthropic,claude-sonnet-4-6,taskset_b_additional_prompts,2,slogan_blood_donation,slogan,diverge,triad,triad_029,triad_029__a2,...,success,One donation. A stranger becomes family.,msg_01NwN8Q4eBjCsGL4LHsevDkQ,end_turn,{'cache_creation': {'ephemeral_1h_input_tokens...,None,r2__7cd39d5a11ce439481e2e8bc,ai_data/deflect_creativity/anthropic/model_cla...,msgbatch_01BAMDhQnbTbSnePARZesaMM,succeeded
1,anthropic,claude-sonnet-4-6,taskset_b_additional_prompts,2,slogan_blood_donation,slogan,diverge,dyad,dyad_060,dyad_060__a2,...,success,One donation. One heartbeat saved.,msg_01QqxHuKdoNjDeieEypkTV3v,end_turn,{'cache_creation': {'ephemeral_1h_input_tokens...,None,r2__062e1dd6324f3d3a83c670da,ai_data/deflect_creativity/anthropic/model_cla...,msgbatch_01BAMDhQnbTbSnePARZesaMM,succeeded
2,anthropic,claude-sonnet-4-6,taskset_b_additional_prompts,2,aut_automobile_tire,aut,vanilla,base,base_081,base_081__a1,...,success,Suspend an automobile tire horizontally from a...,msg_01FaLydVpB6epRHrTWExHaUu,end_turn,{'cache_creation': {'ephemeral_1h_input_tokens...,None,r2__a92ab04391ba220e9bd10f5e,ai_data/deflect_creativity/anthropic/model_cla...,msgbatch_01BAMDhQnbTbSnePARZesaMM,succeeded
3,anthropic,claude-sonnet-4-6,taskset_b_additional_prompts,2,story_life_last_seconds,story,diverge,base,base_036,base_036__a1,...,success,Ezra Bloom was born in a tenement above a tail...,msg_01T3aHbi94vCKxZXKa8sgsiD,end_turn,{'cache_creation': {'ephemeral_1h_input_tokens...,None,r2__63dbdafc35b3ba576c48efb8,ai_data/deflect_creativity/anthropic/model_cla...,msgbatch_01BAMDhQnbTbSnePARZesaMM,succeeded
4,anthropic,claude-sonnet-4-6,taskset_b_additional_prompts,2,story_horror,story,vanilla,base,base_025,base_025__a1,...,success,The old radio at the back of the thrift store ...,msg_01Ph8M5tBtEYR2b3ur6D11XC,end_turn,{'cache_creation': {'ephemeral_1h_input_tokens...,None,r2__5c97edb6fc180fe92c6a4c73,ai_data/deflect_creativity/anthropic/model_cla...,msgbatch_01BAMDhQnbTbSnePARZesaMM,succeeded


In [49]:
# Cell R2-C — Identify failed or empty Round 2 records

round2_bad_df = round2_df[
    ~round2_df["status"].eq("success")
    | round2_df["text"].isna()
    | round2_df["text"].astype(str).str.strip().eq("")
].copy()

print("Bad Round 2 records:", len(round2_bad_df))

display(
    round2_bad_df[[
        "request_key",
        "task_id",
        "task_family",
        "strategy",
        "condition",
        "group_id",
        "agent_id",
        "status",
        "error",
    ]]
)

if len(round2_bad_df) == 0:
    print("No Round 2 repair needed.")
elif len(round2_bad_df) > 20:
    raise RuntimeError("Unexpectedly many failed Round 2 records. Inspect before repairing.")

Bad Round 2 records: 2


,request_key,task_id,task_family,strategy,condition,group_id,agent_id,status,error
3495,r2__30295fcb8dfb5365c3852a13,story_life_last_seconds,story,diverge,dyad,dyad_031,dyad_031__a1,errored,{'error': {'error': {'message': 'Internal Serv...
3887,r2__e71cadebd386325d06c6460b,aut_automobile_tire,aut,vanilla,dyad,dyad_026,dyad_026__a1,errored,{'error': {'error': {'message': 'Internal Serv...


In [50]:
# Cell R2-D — Submit Round 2 repair batch for failed/empty records

def build_anthropic_round2_repair_requests_from_bad_records(
    bad_df: pd.DataFrame,
    original_plan_path: Path,
) -> tuple[list[dict], pd.DataFrame]:
    original_plan_df = pd.read_csv(original_plan_path)

    failed_keys = bad_df["request_key"].dropna().astype(str).tolist()

    repair_plan_df = original_plan_df[
        original_plan_df["request_key"].astype(str).isin(failed_keys)
    ].copy()

    if len(repair_plan_df) != len(failed_keys):
        print("Failed keys:", failed_keys)
        print("Matched repair-plan rows:", len(repair_plan_df))
        raise RuntimeError("Could not match every failed request_key to the original Round 2 plan.")

    repair_plan_df["original_request_key"] = repair_plan_df["request_key"]

    repair_plan_df["request_key"] = repair_plan_df["request_key"].map(
        lambda x: "repair_" + str(x)
    )

    repair_plan_df["repair_of_batch_id"] = round2_batch_info["batch_id"]
    repair_plan_df["repair_created_at_utc"] = now_iso()

    repair_requests = []

    for _, row in repair_plan_df.iterrows():
        repair_requests.append({
            "custom_id": row["request_key"],
            "params": make_anthropic_message_params(row),
        })

    return repair_requests, repair_plan_df


if len(round2_bad_df) > 0:
    round2_repair_requests, round2_repair_plan_df = build_anthropic_round2_repair_requests_from_bad_records(
        bad_df=round2_bad_df,
        original_plan_path=Path(round2_batch_info["plan_path"]),
    )

    timestamp = dt.datetime.now().strftime("%Y%m%d_%H%M%S")

    round2_repair_plan_path = (
        DIRS["round2_plans"]
        / f"{TASK_SET_ID}__round2_repair__{PROVIDER}__{MODEL_NAME}__{timestamp}__plan.csv"
    )

    round2_repair_jsonl_path = (
        DIRS["round2_batch_inputs"]
        / f"{TASK_SET_ID}__round2_repair__{PROVIDER}__{MODEL_NAME}__{timestamp}__local_batch_requests.jsonl"
    )

    if round2_repair_plan_path.exists() or round2_repair_jsonl_path.exists():
        raise FileExistsError("Refusing to overwrite Round 2 repair files.")

    round2_repair_plan_df.to_csv(round2_repair_plan_path, index=False)

    with open(round2_repair_jsonl_path, "w", encoding="utf-8") as f:
        for req in round2_repair_requests:
            f.write(json.dumps(req, ensure_ascii=False) + "\n")

    round2_repair_batch_info = submit_anthropic_batch(
        requests=round2_repair_requests,
        round_name="round2_repair",
        plan_path=round2_repair_plan_path,
        local_jsonl_path=round2_repair_jsonl_path,
        manifest_dir=DIRS["round2_manifests"],
    )

    display(round2_repair_plan_df)
    display(round2_repair_batch_info)

else:
    round2_repair_batch_info = None
    print("No Round 2 repair batch submitted.")

Submitted Anthropic Message Batch:
{
  "run_id": "20260518_095240__3837589d",
  "task_set_id": "taskset_b_additional_prompts",
  "round": "round2_repair",
  "provider": "anthropic",
  "model": "claude-sonnet-4-6",
  "message_batch": {
    "id": "msgbatch_014T3hRTJFPLh8ksjHs18uwN",
    "archived_at": null,
    "cancel_initiated_at": null,
    "created_at": "2026-05-18T15:00:53.355685Z",
    "ended_at": null,
    "expires_at": "2026-05-19T15:00:53.355685Z",
    "processing_status": "in_progress",
    "request_counts": {
      "canceled": 0,
      "errored": 0,
      "expired": 0,
      "processing": 2,
      "succeeded": 0
    },
    "results_url": null,
    "type": "message_batch"
  },
  "batch_id": "msgbatch_014T3hRTJFPLh8ksjHs18uwN",
  "processing_status_at_submission": "in_progress",
  "submitted_at_utc": "2026-05-18T15:00:53.460734+00:00",
  "local_jsonl_path": "ai_data/deflect_creativity/anthropic/model_claude-sonnet-4-6/taskset_b_additional_prompts/run_20260518_095240__3837589d/02

,provider,model,task_set_id,round,task_id,task_family,strategy,condition,group_id,agent_id,...,system_instructions,user_prompt,temperature,max_output_tokens,self_round1_text,peer_round1_texts_json,created_at_utc,original_request_key,repair_of_batch_id,repair_created_at_utc
650,anthropic,claude-sonnet-4-6,taskset_b_additional_prompts,2,aut_automobile_tire,aut,vanilla,dyad,dyad_026,dyad_026__a1,...,You are participating in a controlled creativi...,You are participating in a creativity task.\n\...,1.0,120,Stack several automobile tires and fill them w...,"[""Stack several tires vertically and fill them...",2026-05-18T14:42:17.169099+00:00,r2__e71cadebd386325d06c6460b,msgbatch_01BAMDhQnbTbSnePARZesaMM,2026-05-18T15:00:52.719331+00:00
4710,anthropic,claude-sonnet-4-6,taskset_b_additional_prompts,2,story_life_last_seconds,story,diverge,dyad,dyad_031,dyad_031__a1,...,You are participating in a controlled creativi...,You are participating in a creative writing ta...,1.0,700,Mara Chen had spent a century as a cartographe...,"[""Mara Solano spent a century chasing storms —...",2026-05-18T14:42:18.187818+00:00,r2__30295fcb8dfb5365c3852a13,msgbatch_01BAMDhQnbTbSnePARZesaMM,2026-05-18T15:00:52.719331+00:00


{'run_id': '20260518_095240__3837589d',
 'task_set_id': 'taskset_b_additional_prompts',
 'round': 'round2_repair',
 'provider': 'anthropic',
 'model': 'claude-sonnet-4-6',
 'message_batch': {'id': 'msgbatch_014T3hRTJFPLh8ksjHs18uwN',
  'archived_at': None,
  'cancel_initiated_at': None,
  'created_at': '2026-05-18T15:00:53.355685Z',
  'ended_at': None,
  'expires_at': '2026-05-19T15:00:53.355685Z',
  'processing_status': 'in_progress',
  'request_counts': {'canceled': 0,
   'errored': 0,
   'expired': 0,
   'processing': 2,
   'succeeded': 0},
  'results_url': None,
  'type': 'message_batch'},
 'batch_id': 'msgbatch_014T3hRTJFPLh8ksjHs18uwN',
 'processing_status_at_submission': 'in_progress',
 'submitted_at_utc': '2026-05-18T15:00:53.460734+00:00',
 'local_jsonl_path': 'ai_data/deflect_creativity/anthropic/model_claude-sonnet-4-6/taskset_b_additional_prompts/run_20260518_095240__3837589d/02_round2/batch_inputs/taskset_b_additional_prompts__round2_repair__anthropic__claude-sonnet-4-6__2

In [53]:
# Cell R2-E — Check Round 2 repair batch status

if round2_repair_batch_info is not None:
    round2_repair_status = check_anthropic_batch(round2_repair_batch_info["batch_id"])
    display(round2_repair_status)
else:
    print("No Round 2 repair batch exists.")

{
  "id": "msgbatch_014T3hRTJFPLh8ksjHs18uwN",
  "archived_at": null,
  "cancel_initiated_at": null,
  "created_at": "2026-05-18T15:00:53.355685Z",
  "ended_at": "2026-05-18T15:02:47.347904Z",
  "expires_at": "2026-05-19T15:00:53.355685Z",
  "processing_status": "ended",
  "request_counts": {
    "canceled": 0,
    "errored": 0,
    "expired": 0,
    "processing": 0,
    "succeeded": 2
  },
  "results_url": "https://api.anthropic.com/v1/messages/batches/msgbatch_014T3hRTJFPLh8ksjHs18uwN/results",
  "type": "message_batch"
}


{'id': 'msgbatch_014T3hRTJFPLh8ksjHs18uwN',
 'archived_at': None,
 'cancel_initiated_at': None,
 'created_at': '2026-05-18T15:00:53.355685Z',
 'ended_at': '2026-05-18T15:02:47.347904Z',
 'expires_at': '2026-05-19T15:00:53.355685Z',
 'processing_status': 'ended',
 'request_counts': {'canceled': 0,
  'errored': 0,
  'expired': 0,
  'processing': 0,
  'succeeded': 2},
 'results_url': 'https://api.anthropic.com/v1/messages/batches/msgbatch_014T3hRTJFPLh8ksjHs18uwN/results',
 'type': 'message_batch'}

In [54]:
# Cell R2-F — Download/reuse and parse Round 2 repair batch

if round2_repair_batch_info is not None:
    round2_repair_output_path = (
        DIRS["round2_raw_outputs"]
        / f"{TASK_SET_ID}__round2_repair__{round2_repair_batch_info['batch_id']}__results.jsonl"
    )

    if round2_repair_output_path.exists():
        print("Using existing Round 2 repair raw output file:")
        print(round2_repair_output_path)
    else:
        round2_repair_output_path = download_anthropic_batch_results(
            batch_id=round2_repair_batch_info["batch_id"],
            raw_output_dir=DIRS["round2_raw_outputs"],
            round_name="round2_repair",
        )

    existing_repair_parsed_pkls = sorted(
        DIRS["round2_parsed"].glob(
            f"{TASK_SET_ID}__round2_repair__{PROVIDER}__{MODEL_NAME}__{round2_repair_batch_info['batch_id']}__parsed.pkl"
        )
    )

    if existing_repair_parsed_pkls:
        round2_repair_pkl_path = existing_repair_parsed_pkls[-1]
        print("Loading existing parsed Round 2 repair file:")
        print(round2_repair_pkl_path)
        round2_repair_df = pd.read_pickle(round2_repair_pkl_path)

    else:
        round2_repair_parse_summary = parse_anthropic_batch_output_to_standard_files(
            batch_output_path=round2_repair_output_path,
            plan_path=Path(round2_repair_batch_info["plan_path"]),
            parsed_dir=DIRS["round2_parsed"],
            round_name="round2_repair",
            batch_id=round2_repair_batch_info["batch_id"],
        )

        round2_repair_df = pd.read_pickle(round2_repair_parse_summary["parsed_pkl_path"])

    print(round2_repair_df.shape)
    display(round2_repair_df["status"].value_counts(dropna=False))

    display(round2_repair_df[[
        "request_key",
        "original_request_key",
        "task_id",
        "strategy",
        "condition",
        "group_id",
        "agent_id",
        "status",
        "text",
        "error",
    ]])

else:
    round2_repair_df = pd.DataFrame()
    print("No Round 2 repair batch to parse.")

Downloaded/streamed 2 results to: ai_data/deflect_creativity/anthropic/model_claude-sonnet-4-6/taskset_b_additional_prompts/run_20260518_095240__3837589d/02_round2/raw_outputs/taskset_b_additional_prompts__round2_repair__msgbatch_014T3hRTJFPLh8ksjHs18uwN__results.jsonl
{
  "task_set_id": "taskset_b_additional_prompts",
  "round": "round2_repair",
  "batch_id": "msgbatch_014T3hRTJFPLh8ksjHs18uwN",
  "n_records": 2,
  "n_success": 2,
  "n_empty_text": 0,
  "n_error": 0,
  "n_canceled": 0,
  "n_expired": 0,
  "parsed_jsonl_path": "ai_data/deflect_creativity/anthropic/model_claude-sonnet-4-6/taskset_b_additional_prompts/run_20260518_095240__3837589d/02_round2/parsed/taskset_b_additional_prompts__round2_repair__anthropic__claude-sonnet-4-6__msgbatch_014T3hRTJFPLh8ksjHs18uwN__parsed.jsonl",
  "parsed_csv_path": "ai_data/deflect_creativity/anthropic/model_claude-sonnet-4-6/taskset_b_additional_prompts/run_20260518_095240__3837589d/02_round2/parsed/taskset_b_additional_prompts__round2_repair__

status
success    2
Name: count, dtype: int64

,request_key,original_request_key,task_id,strategy,condition,group_id,agent_id,status,text,error
0,repair_r2__e71cadebd386325d06c6460b,r2__e71cadebd386325d06c6460b,aut_automobile_tire,vanilla,dyad,dyad_026,dyad_026__a1,success,Suspend an automobile tire from a sturdy tree ...,None
1,repair_r2__30295fcb8dfb5365c3852a13,r2__30295fcb8dfb5365c3852a13,story_life_last_seconds,diverge,dyad,dyad_031,dyad_031__a1,success,Ezekiel Barrow had lived one hundred years as ...,None


In [55]:
# Cell R2-G — Merge repaired Round 2 records and save repaired Round 2 file

def merge_round2_with_repairs(
    original_df: pd.DataFrame,
    repair_df: pd.DataFrame,
) -> pd.DataFrame:
    if repair_df.empty:
        return original_df.copy()

    repair_success = repair_df[repair_df["status"].eq("success")].copy()

    if len(repair_success) != len(repair_df):
        display(repair_df[repair_df["status"] != "success"])
        raise RuntimeError("At least one Round 2 repair request failed. Inspect before proceeding.")

    if "original_request_key" not in repair_success.columns:
        raise RuntimeError("Repair dataframe is missing original_request_key.")

    repaired_df = original_df.copy()

    for _, repaired_row in repair_success.iterrows():
        original_key = repaired_row["original_request_key"]

        mask = repaired_df["request_key"].astype(str).eq(str(original_key))

        if mask.sum() != 1:
            raise RuntimeError(f"Could not uniquely locate original failed Round 2 row: {original_key}")

        replacement = repaired_row.copy()
        replacement["repair_request_key"] = repaired_row["request_key"]
        replacement["request_key"] = original_key
        replacement["repaired_from_batch_id"] = repaired_row.get("batch_id")
        replacement["repair_applied_at_utc"] = now_iso()

        for col in replacement.index:
            if col not in repaired_df.columns:
                repaired_df[col] = None

        repaired_df.loc[mask, replacement.index] = replacement.values

    return repaired_df


round2_df_repaired = merge_round2_with_repairs(
    original_df=round2_df,
    repair_df=round2_repair_df,
)

print(round2_df_repaired.shape)
display(round2_df_repaired["status"].value_counts(dropna=False))

round2_remaining_bad = round2_df_repaired[
    ~round2_df_repaired["status"].eq("success")
    | round2_df_repaired["text"].isna()
    | round2_df_repaired["text"].astype(str).str.strip().eq("")
].copy()

print("Remaining bad Round 2 records:", len(round2_remaining_bad))

display(round2_remaining_bad[[
    "request_key",
    "task_id",
    "strategy",
    "condition",
    "group_id",
    "agent_id",
    "status",
    "error",
]])

expected_round2 = len(TASK_SETTINGS) * len(STRATEGIES) * (N_BASE_AGENTS + 2 * N_DYADS + 3 * N_TRIADS)

print("Expected Round 2 rows:", expected_round2)
print("Actual Round 2 rows:  ", len(round2_df_repaired))

if len(round2_df_repaired) != expected_round2:
    raise RuntimeError("Round 2 repaired row count does not match expectation.")

if len(round2_remaining_bad) > 0:
    raise RuntimeError("Round 2 still has failed or empty records after repair.")

timestamp = dt.datetime.now().strftime("%Y%m%d_%H%M%S")

round2_repaired_csv_path = (
    DIRS["round2_parsed"]
    / f"{TASK_SET_ID}__round2__{PROVIDER}__{MODEL_NAME}__repaired__{timestamp}.csv"
)

round2_repaired_pkl_path = (
    DIRS["round2_parsed"]
    / f"{TASK_SET_ID}__round2__{PROVIDER}__{MODEL_NAME}__repaired__{timestamp}.pkl"
)

if round2_repaired_csv_path.exists() or round2_repaired_pkl_path.exists():
    raise FileExistsError("Refusing to overwrite repaired Round 2 files.")

round2_df_repaired.to_csv(round2_repaired_csv_path, index=False)
round2_df_repaired.to_pickle(round2_repaired_pkl_path)

print("Saved repaired Round 2 files:")
print(round2_repaired_csv_path)
print(round2_repaired_pkl_path)

round2_df = round2_df_repaired

(5400, 38)


status
success    5400
Name: count, dtype: int64

Remaining bad Round 2 records: 0


,request_key,task_id,strategy,condition,group_id,agent_id,status,error


Expected Round 2 rows: 5400
Actual Round 2 rows:   5400
Saved repaired Round 2 files:
ai_data/deflect_creativity/anthropic/model_claude-sonnet-4-6/taskset_b_additional_prompts/run_20260518_095240__3837589d/02_round2/parsed/taskset_b_additional_prompts__round2__anthropic__claude-sonnet-4-6__repaired__20260518_110444.csv
ai_data/deflect_creativity/anthropic/model_claude-sonnet-4-6/taskset_b_additional_prompts/run_20260518_095240__3837589d/02_round2/parsed/taskset_b_additional_prompts__round2__anthropic__claude-sonnet-4-6__repaired__20260518_110444.pkl


In [ ]:
# Optional reload after kernel restart:
# manifest_path = Path("ai_data/deflect_creativity/anthropic/model_claude-sonnet-4-6/taskset_b_additional_prompts/run_.../02_round2/manifests/...")
# round2_batch_info = read_json(manifest_path)
# DATA_ROOT = Path(round2_batch_info["data_root"])
# round2_batch_info

In [ ]:
# round2_output_path = download_anthropic_batch_results(
#     batch_id=round2_batch_info["batch_id"],
#     raw_output_dir=DIRS["round2_raw_outputs"],
#     round_name="round2",
# )

# round2_output_path

In [ ]:
# round2_parse_summary = parse_anthropic_batch_output_to_standard_files(
#     batch_output_path=round2_output_path,
#     plan_path=Path(round2_batch_info["plan_path"]),
#     parsed_dir=DIRS["round2_parsed"],
#     round_name="round2",
#     batch_id=round2_batch_info["batch_id"],
# )

# round2_df = pd.read_pickle(round2_parse_summary["parsed_pkl_path"])
# print(round2_df.shape)
# display(round2_df["status"].value_counts(dropna=False))
# round2_df.head()

In [56]:
expected_round2 = expected_round1
actual_round2 = len(round2_df)

print("Expected Round 2 rows:", expected_round2)
print("Actual Round 2 rows:  ", actual_round2)

display(round2_df.groupby(["task_id", "strategy", "condition", "status"]).size().reset_index(name="n"))

if actual_round2 != expected_round2:
    print("WARNING: Row count mismatch. Inspect errors before compiling.")

if (round2_df["status"] != "success").any():
    print("WARNING: Some Round 2 calls failed or returned empty text.")
    display(round2_df[round2_df["status"] != "success"].head(20))
else:
    print("Round 2 looks complete.")

Expected Round 2 rows: 5400
Actual Round 2 rows:   5400


,task_id,strategy,condition,status,n
0,aut_automobile_tire,diverge,base,success,150
1,aut_automobile_tire,diverge,dyad,success,150
2,aut_automobile_tire,diverge,triad,success,150
3,aut_automobile_tire,vanilla,base,success,150
4,aut_automobile_tire,vanilla,dyad,success,150
5,aut_automobile_tire,vanilla,triad,success,150
6,aut_key,diverge,base,success,150
7,aut_key,diverge,dyad,success,150
8,aut_key,diverge,triad,success,150
9,aut_key,vanilla,base,success,150


Round 2 looks complete.


In [57]:
def normalize_for_analysis(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()

    for col in [
        "self_round1_request_key",
        "peer_round1_agent_ids",
        "self_round1_text",
        "peer_round1_texts_json",
    ]:
        if col not in out.columns:
            out[col] = None

    keep_cols = [
        "provider",
        "model",
        "task_set_id",
        "round",
        "task_id",
        "task_family",
        "strategy",
        "condition",
        "group_id",
        "agent_id",
        "agent_index",
        "request_key",
        "status",
        "text",
        "temperature",
        "max_output_tokens",
        "self_round1_request_key",
        "peer_round1_agent_ids",
        "self_round1_text",
        "peer_round1_texts_json",
        "provider_response_id",
        "stop_reason",
        "usage",
        "error",
        "batch_id",
        "batch_custom_id",
        "batch_output_file",
        "parsed_at_utc",
    ]

    existing_keep_cols = [c for c in keep_cols if c in out.columns]
    out = out[existing_keep_cols].copy()

    out["text_clean"] = out["text"].map(clean_model_text)
    out["is_success"] = out["status"].eq("success")

    return out


round1_analysis_df = normalize_for_analysis(round1_df)
round2_analysis_df = normalize_for_analysis(round2_df)

full_long_df = pd.concat([round1_analysis_df, round2_analysis_df], ignore_index=True)

sort_cols = ["task_id", "strategy", "condition", "group_id", "agent_index", "round"]
full_long_df = full_long_df.sort_values(sort_cols).reset_index(drop=True)

print(full_long_df.shape)
display(full_long_df.groupby(["round", "task_id", "strategy", "condition", "status"]).size().reset_index(name="n").head(40))

timestamp = dt.datetime.now().strftime("%Y%m%d_%H%M%S")

full_csv_path = DIRS["compiled"] / f"{TASK_SET_ID}__{PROVIDER}__{MODEL_NAME}__deflect_creativity__full_long__{timestamp}.csv"
full_pkl_path = DIRS["compiled"] / f"{TASK_SET_ID}__{PROVIDER}__{MODEL_NAME}__deflect_creativity__full_long__{timestamp}.pkl"

if full_csv_path.exists() or full_pkl_path.exists():
    raise FileExistsError("Refusing to overwrite compiled long files.")

full_long_df.to_csv(full_csv_path, index=False)
full_long_df.to_pickle(full_pkl_path)

print(full_csv_path)
print(full_pkl_path)

(10800, 30)


,round,task_id,strategy,condition,status,n
0,1,aut_automobile_tire,diverge,base,success,150
1,1,aut_automobile_tire,diverge,dyad,success,150
2,1,aut_automobile_tire,diverge,triad,success,150
3,1,aut_automobile_tire,vanilla,base,success,150
4,1,aut_automobile_tire,vanilla,dyad,success,150
5,1,aut_automobile_tire,vanilla,triad,success,150
6,1,aut_key,diverge,base,success,150
7,1,aut_key,diverge,dyad,success,150
8,1,aut_key,diverge,triad,success,150
9,1,aut_key,vanilla,base,success,150


ai_data/deflect_creativity/anthropic/model_claude-sonnet-4-6/taskset_b_additional_prompts/run_20260518_095240__3837589d/03_compiled/taskset_b_additional_prompts__anthropic__claude-sonnet-4-6__deflect_creativity__full_long__20260518_110500.csv
ai_data/deflect_creativity/anthropic/model_claude-sonnet-4-6/taskset_b_additional_prompts/run_20260518_095240__3837589d/03_compiled/taskset_b_additional_prompts__anthropic__claude-sonnet-4-6__deflect_creativity__full_long__20260518_110500.pkl


In [58]:
r1_small = full_long_df[full_long_df["round"].eq(1)].copy()
r2_small = full_long_df[full_long_df["round"].eq(2)].copy()

merge_keys = [
    "provider",
    "model",
    "task_set_id",
    "task_id",
    "task_family",
    "strategy",
    "condition",
    "group_id",
    "agent_id",
    "agent_index",
]

wide_df = r1_small[merge_keys + ["request_key", "status", "text_clean", "batch_id", "usage"]].rename(
    columns={
        "request_key": "round1_request_key",
        "status": "round1_status",
        "text_clean": "round1_text",
        "batch_id": "round1_batch_id",
        "usage": "round1_usage",
    }
).merge(
    r2_small[merge_keys + [
        "request_key",
        "status",
        "text_clean",
        "batch_id",
        "usage",
        "self_round1_request_key",
        "peer_round1_agent_ids",
        "self_round1_text",
        "peer_round1_texts_json",
    ]].rename(
        columns={
            "request_key": "round2_request_key",
            "status": "round2_status",
            "text_clean": "round2_text",
            "batch_id": "round2_batch_id",
            "usage": "round2_usage",
        }
    ),
    on=merge_keys,
    how="outer",
    validate="one_to_one",
)

wide_df = wide_df.sort_values(["task_id", "strategy", "condition", "group_id", "agent_index"]).reset_index(drop=True)

wide_csv_path = DIRS["compiled"] / f"{TASK_SET_ID}__{PROVIDER}__{MODEL_NAME}__deflect_creativity__ego_wide__{timestamp}.csv"
wide_pkl_path = DIRS["compiled"] / f"{TASK_SET_ID}__{PROVIDER}__{MODEL_NAME}__deflect_creativity__ego_wide__{timestamp}.pkl"

if wide_csv_path.exists() or wide_pkl_path.exists():
    raise FileExistsError("Refusing to overwrite compiled wide files.")

wide_df.to_csv(wide_csv_path, index=False)
wide_df.to_pickle(wide_pkl_path)

print(wide_df.shape)
print(wide_csv_path)
print(wide_pkl_path)
wide_df.head()

(5400, 24)
ai_data/deflect_creativity/anthropic/model_claude-sonnet-4-6/taskset_b_additional_prompts/run_20260518_095240__3837589d/03_compiled/taskset_b_additional_prompts__anthropic__claude-sonnet-4-6__deflect_creativity__ego_wide__20260518_110500.csv
ai_data/deflect_creativity/anthropic/model_claude-sonnet-4-6/taskset_b_additional_prompts/run_20260518_095240__3837589d/03_compiled/taskset_b_additional_prompts__anthropic__claude-sonnet-4-6__deflect_creativity__ego_wide__20260518_110500.pkl


,provider,model,task_set_id,task_id,task_family,strategy,condition,group_id,agent_id,agent_index,...,round1_usage,round2_request_key,round2_status,round2_text,round2_batch_id,round2_usage,self_round1_request_key,peer_round1_agent_ids,self_round1_text,peer_round1_texts_json
0,anthropic,claude-sonnet-4-6,taskset_b_additional_prompts,aut_automobile_tire,aut,diverge,base,base_001,base_001__a1,1,...,{'cache_creation': {'ephemeral_1h_input_tokens...,r2__668c1941d0a0da8f68ddd2f2,success,Hung vertically from a dock piling as a cushio...,msgbatch_01BAMDhQnbTbSnePARZesaMM,{'cache_creation': {'ephemeral_1h_input_tokens...,r1__ed4b70c212cc324219b1d60f,NaN,"Stacked and filled with soil, used as a raised...",[]
1,anthropic,claude-sonnet-4-6,taskset_b_additional_prompts,aut_automobile_tire,aut,diverge,base,base_002,base_002__a1,1,...,{'cache_creation': {'ephemeral_1h_input_tokens...,r2__8a94adc496b363405ff863ba,success,"The rubber sidewall, cut into strips and woven...",msgbatch_01BAMDhQnbTbSnePARZesaMM,{'cache_creation': {'ephemeral_1h_input_tokens...,r1__943c81b2745feadd05d09e9f,NaN,Strung horizontally between two trees with rop...,[]
2,anthropic,claude-sonnet-4-6,taskset_b_additional_prompts,aut_automobile_tire,aut,diverge,base,base_003,base_003__a1,1,...,{'cache_creation': {'ephemeral_1h_input_tokens...,r2__22f0b547a3160fe3322da286,success,Stack several tires vertically and fill them w...,msgbatch_01BAMDhQnbTbSnePARZesaMM,{'cache_creation': {'ephemeral_1h_input_tokens...,r1__47e676e0a2df1bfb1e9d9183,NaN,Suspend a tire vertically from a sturdy tree b...,[]
3,anthropic,claude-sonnet-4-6,taskset_b_additional_prompts,aut_automobile_tire,aut,diverge,base,base_004,base_004__a1,1,...,{'cache_creation': {'ephemeral_1h_input_tokens...,r2__58f14167e3d1a442ec1b5942,success,"Sliced into thin cross-sectional rings, the ti...",msgbatch_01BAMDhQnbTbSnePARZesaMM,{'cache_creation': {'ephemeral_1h_input_tokens...,r1__7b6deb386a9b3a84c8ddcbd1,NaN,Strung horizontally between two trees with a p...,[]
4,anthropic,claude-sonnet-4-6,taskset_b_additional_prompts,aut_automobile_tire,aut,diverge,base,base_005,base_005__a1,1,...,{'cache_creation': {'ephemeral_1h_input_tokens...,r2__783bd644bd15acd2312e9ea8,success,Suspend a tire horizontally from a sturdy beam...,msgbatch_01BAMDhQnbTbSnePARZesaMM,{'cache_creation': {'ephemeral_1h_input_tokens...,r1__c4f867e1d900be5898fbf715,NaN,Stack several automobile tires vertically and ...,[]


In [59]:
def word_count(text: str) -> int:
    if not isinstance(text, str):
        return 0
    return len(re.findall(r"\b[\w'-]+\b", text))


def sentence_count_rough(text: str) -> int:
    if not isinstance(text, str):
        return 0
    parts = re.split(r"(?<=[.!?])\s+", text.strip())
    parts = [p for p in parts if p.strip()]
    return len(parts)


validation_df = full_long_df.copy()
validation_df["word_count"] = validation_df["text_clean"].map(word_count)
validation_df["rough_sentence_count"] = validation_df["text_clean"].map(sentence_count_rough)

slogan_violations = validation_df[
    validation_df["task_family"].eq("slogan")
    & validation_df["is_success"]
    & validation_df["word_count"].gt(6)
].copy()

story_sentence_violations = validation_df[
    validation_df["task_family"].eq("story")
    & validation_df["is_success"]
    & validation_df["rough_sentence_count"].ne(8)
].copy()

print("Slogan >6-word violations:", len(slogan_violations))
display(slogan_violations[["round", "task_id", "strategy", "condition", "agent_id", "text_clean", "word_count"]].head(20))

print("Story rough sentence-count violations:", len(story_sentence_violations))
display(story_sentence_violations[["round", "task_id", "strategy", "condition", "agent_id", "text_clean", "rough_sentence_count"]].head(20))

validation_csv_path = DIRS["compiled"] / f"{TASK_SET_ID}__{PROVIDER}__{MODEL_NAME}__deflect_creativity__validation_flags__{timestamp}.csv"

if validation_csv_path.exists():
    raise FileExistsError(f"Refusing to overwrite validation file: {validation_csv_path}")

validation_df.to_csv(validation_csv_path, index=False)
validation_csv_path

Slogan >6-word violations: 50


,round,task_id,strategy,condition,agent_id,text_clean,word_count
5437,2,slogan_blood_donation,diverge,base,base_019__a1,"One vein at a time, save lives.",7
5485,2,slogan_blood_donation,diverge,base,base_043__a1,"One vein at a time, save lives.",7
5491,2,slogan_blood_donation,diverge,base,base_046__a1,Strangers share life one drop at a time.,8
5518,1,slogan_blood_donation,diverge,base,base_060__a1,Your spare pint saves a whole story.,7
5524,1,slogan_blood_donation,diverge,base,base_063__a1,Your blood runs through a stranger's story.,7
5553,2,slogan_blood_donation,diverge,base,base_077__a1,Gaps in the blood supply cost lives.,7
5581,2,slogan_blood_donation,diverge,base,base_091__a1,"One vein at a time, save lives.",7
5585,2,slogan_blood_donation,diverge,base,base_093__a1,"One vein at a time, save lives.",7
5587,2,slogan_blood_donation,diverge,base,base_094__a1,"One vein at a time, save someone.",7
5668,1,slogan_blood_donation,diverge,base,base_135__a1,Your blood runs through someone else's story.,7


Story rough sentence-count violations: 1326


,round,task_id,strategy,condition,agent_id,text_clean,rough_sentence_count
7200,1,story_horror,diverge,base,base_001__a1,"The night Maya moved into her college dorm, sh...",7
7204,1,story_horror,diverge,base,base_003__a1,"The summer my grandmother died, she left me he...",9
7205,2,story_horror,diverge,base,base_003__a1,The photographs in my family's albums have alw...,7
7206,1,story_horror,diverge,base,base_004__a1,The search party found her journal half-buried...,7
7207,2,story_horror,diverge,base,base_004__a1,"The voicemail came at 3:14 a.m., a full six ho...",7
7209,2,story_horror,diverge,base,base_005__a1,The notification appeared at 3:14 a.m.: a voic...,6
7210,1,story_horror,diverge,base,base_006__a1,The first photograph in the album shows my gra...,7
7212,1,story_horror,diverge,base,base_007__a1,The voicemail from my mother lasted three minu...,9
7214,1,story_horror,diverge,base,base_008__a1,The mirror in my grandmother's old house had a...,7
7215,2,story_horror,diverge,base,base_008__a1,"The summer my older brother drowned, we never ...",9


PosixPath('ai_data/deflect_creativity/anthropic/model_claude-sonnet-4-6/taskset_b_additional_prompts/run_20260518_095240__3837589d/03_compiled/taskset_b_additional_prompts__anthropic__claude-sonnet-4-6__deflect_creativity__validation_flags__20260518_110500.csv')

In [60]:
final_manifest = {
    "run_id": RUN_ID,
    "task_set_id": TASK_SET_ID,
    "data_root": str(DATA_ROOT),
    "provider": PROVIDER,
    "model": MODEL_NAME,
    "round1_batch_info": round1_batch_info,
    "round2_batch_info": round2_batch_info,
    "round1_parse_summary": round1_parse_summary,
    "round2_parse_summary": round2_parse_summary,
    "compiled_long_csv": str(full_csv_path),
    "compiled_long_pkl": str(full_pkl_path),
    "compiled_wide_csv": str(wide_csv_path),
    "compiled_wide_pkl": str(wide_pkl_path),
    "validation_csv": str(validation_csv_path),
    "completed_at_utc": now_iso(),
}

final_manifest_path = DIRS["compiled"] / f"{TASK_SET_ID}__{PROVIDER}__{MODEL_NAME}__deflect_creativity__final_manifest__{timestamp}.json"

if final_manifest_path.exists():
    raise FileExistsError(f"Refusing to overwrite final manifest: {final_manifest_path}")

write_json(final_manifest_path, final_manifest)

final_manifest_path

PosixPath('ai_data/deflect_creativity/anthropic/model_claude-sonnet-4-6/taskset_b_additional_prompts/run_20260518_095240__3837589d/03_compiled/taskset_b_additional_prompts__anthropic__claude-sonnet-4-6__deflect_creativity__final_manifest__20260518_110500.json')